# 01 - Data Audit

## Purpose

This notebook validates the raw datasets used for the Milestone II project before cleaning, feature engineering, EDA, or modeling.

### Goals
1. Verify that all required source files can be loaded.
2. Document dataset dimensions, fields, years, and geographic coverage.
3. Standardize county FIPS codes across sources.
4. Identify duplicates, missing values, and unmatched counties.
5. Isolate the CDC PLACES routine-checkup target.
6. Inspect candidate AHRF, HPSA, and SVI variables.
7. Test incremental county-level joins.
8. Save an initial harmonized dataset for downstream analysis.

### Core data sources
- CDC PLACES 2024
- HRSA Area Health Resources Files (AHRF) 2023–2024
- HRSA HPSA Primary Care, Mental Health, and Dental Health
- CDC/ATSDR Social Vulnerability Index (SVI) 2022

---

## 1. Setup and Project Paths

In [1]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd

In [2]:
from pathlib import Path

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
REFERENCE_DIR = DATA_DIR / "reference"
INTERIM_DIR = DATA_DIR / "interim"

PLACES_DIR = RAW_DIR / "places"
AHRF_DIR = RAW_DIR / "ahrf"
HPSA_DIR = RAW_DIR / "hpsa"
SVI_DIR = RAW_DIR / "svi"

INTERIM_DIR.mkdir(parents = True, exist_ok = True)

---

## 2. Source File Inventory

This section confirms that the expected raw and reference files are available before any data are loaded or modified.

In [3]:
folders = {
    "PLACES": PLACES_DIR,
    "AHRF": AHRF_DIR,
    "HPSA": HPSA_DIR,
    "SVI": SVI_DIR,
    "Reference": REFERENCE_DIR,
}

for name, folder in folders.items():
    print(f"\n{name}: {folder}")

    if folder.exists():
        files = list(folder.iterdir())

        if files:
            for file in files:
                print("  -", file.name)
        else:
            print("  [folder is empty]")
    else:
        print("  [folder not found]")


PLACES: ../data/raw/places
  - places_county_2024.csv

AHRF: ../data/raw/ahrf
  - ahrf_county_2023_2024.zip

HPSA: ../data/raw/hpsa
  - hpsa_primary_care_2026-09-25.csv
  - hpsa_mental_health_2026-09-25.csv
  - hpsa_dental_health_2026-09-25.csv

SVI: ../data/raw/svi
  - svi_county_2022.csv

Reference: ../data/reference
  - feature_manifest.csv
  - ahrf_candidate_variables_team17.csv


In [4]:
print("Current directory:", Path.cwd())
print("Project root:", PROJECT_ROOT.resolve())

Current directory: /voc/work/milestone2-team17/notebooks
Project root: /voc/work/milestone2-team17


---

## 3. CDC PLACES Audit

CDC PLACES provides the supervised-learning outcome for this project. This section inspects the raw 2024 county release and identifies the county-level routine-checkup measure used as the project target.

In [5]:
import numpy as np
import pandas as pd

In [6]:
places_path = PLACES_DIR / "places_county_2024.csv"

places = pd.read_csv(
    places_path,
    low_memory = False
)

print("PLACES shape:", places.shape)

PLACES shape: (240886, 22)


In [7]:
places.head()

,Year,StateAbbr,StateDesc,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,...,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text,Geolocation
0,2022,AL,Alabama,Clay,BRFSS,Health Outcomes,Current asthma among adults,%,Crude prevalence,11.0,...,9.6,12.4,"14,198","11,235",1027,HLTHOUT,CASTHMA,CrdPrv,Current Asthma,POINT (-85.8606604130173 33.2693085517833)
1,2022,AL,Alabama,Dale,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,35.6,...,34.8,36.4,"49,544","38,039",1045,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-85.6109266826185 31.4316629601359)
2,2022,AL,Alabama,Jackson,BRFSS,Health Outcomes,Stroke among adults,%,Crude prevalence,5.1,...,4.6,5.6,"52,891","42,087",1071,HLTHOUT,STROKE,CrdPrv,Stroke,POINT (-85.9995052603961 34.7796392771775)
3,2022,AL,Alabama,Lauderdale,BRFSS,Health Outcomes,Obesity among adults,%,Age-adjusted prevalence,36.5,...,29.6,43.9,"95,878","77,472",1077,HLTHOUT,OBESITY,AgeAdjPrv,Obesity,POINT (-87.6540983850983 34.9014438477101)
4,2022,AL,Alabama,Lawrence,BRFSS,Disability,Any disability among adults,%,Crude prevalence,40.1,...,35.5,44.7,"33,214","26,022",1079,DISABLT,DISABILITY,CrdPrv,Any Disability,POINT (-87.3108851040374 34.5216735395968)


In [8]:
places.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240886 entries, 0 to 240885
Data columns (total 22 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Year                        240886 non-null  int64  
 1   StateAbbr                   240886 non-null  object 
 2   StateDesc                   240886 non-null  object 
 3   LocationName                240806 non-null  object 
 4   DataSource                  240886 non-null  object 
 5   Category                    240886 non-null  object 
 6   Measure                     240886 non-null  object 
 7   Data_Value_Unit             240886 non-null  object 
 8   Data_Value_Type             240886 non-null  object 
 9   Data_Value                  240886 non-null  float64
 10  Data_Value_Footnote_Symbol  14 non-null      object 
 11  Data_Value_Footnote         14 non-null      object 
 12  Low_Confidence_Limit        240886 non-null  float64
 13  High_Confidenc

In [9]:
# inspect the columns
places.columns.tolist()

['Year',
 'StateAbbr',
 'StateDesc',
 'LocationName',
 'DataSource',
 'Category',
 'Measure',
 'Data_Value_Unit',
 'Data_Value_Type',
 'Data_Value',
 'Data_Value_Footnote_Symbol',
 'Data_Value_Footnote',
 'Low_Confidence_Limit',
 'High_Confidence_Limit',
 'TotalPopulation',
 'TotalPop18plus',
 'LocationID',
 'CategoryID',
 'MeasureId',
 'DataValueTypeID',
 'Short_Question_Text',
 'Geolocation']

In [10]:
# simple audit stats
print("Number of rows:", len(places))
print("Number of columns:", places.shape[1])
print("Duplicate full rows:", places.duplicated().sum())

Number of rows: 240886
Number of columns: 22
Duplicate full rows: 0


In [11]:
# missingness
places.isna().sum().sort_values(ascending = False).head(15)

Data_Value_Footnote_Symbol    240872
Data_Value_Footnote           240872
LocationName                      80
Geolocation                       80
StateAbbr                          0
Year                               0
Category                           0
DataSource                         0
StateDesc                          0
Measure                            0
Data_Value                         0
Data_Value_Type                    0
Low_Confidence_Limit               0
Data_Value_Unit                    0
High_Confidence_Limit              0
dtype: int64

### 3.1 Identify the Routine-Checkup Target

The supervised-learning target is the age-adjusted county prevalence of adults reporting a routine checkup within the past year.

In [12]:
# before filtering we want to confirm how the measure is labeled
routine_measures = (
    places.loc[
        places["Measure"].str.contains(
            "routine checkup",
            case = False,
            na = False
        ),
        "Measure"
    ]
    .drop_duplicates()
    .sort_values()
)

routine_measures

615    Visits to doctor for routine checkup within th...
Name: Measure, dtype: object

In [13]:
# inspect the value types associated w/ routine check-up measure
places.loc[
    places["Measure"].str.contains(
        "routine checkup",
        case = False,
        na = False
    ),
    "Data_Value_Type"
].value_counts()

Data_Value_Type
Age-adjusted prevalence    3145
Crude prevalence           3145
Name: count, dtype: int64

In [14]:
# filter for age-altered
places_target = places[
    places["Measure"].str.contains(
        "routine checkup",
        case = False,
        na = False
    )
    &
    places["Data_Value_Type"].str.contains(
        "Age-adjusted",
        case = False,
        na = False
    )
].copy()

In [15]:
# sanity check
print("Target rows:", len(places_target))
places_target.head()

Target rows: 3145


,Year,StateAbbr,StateDesc,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,...,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text,Geolocation
615,2022,CA,California,Mendocino,BRFSS,Prevention,Visits to doctor for routine checkup within th...,%,Age-adjusted prevalence,68.6,...,64.0,72.9,"89,783","71,066",6045,PREVENT,CHECKUP,AgeAdjPrv,Annual Checkup,POINT (-123.391321827245 39.4403114811983)
736,2022,AK,Alaska,Chugach,BRFSS,Prevention,Visits to doctor for routine checkup within th...,%,Age-adjusted prevalence,62.2,...,57.6,67.0,"6,874","5,338",2063,PREVENT,CHECKUP,AgeAdjPrv,Annual Checkup,POINT (-145.846670254475 60.7148095406379)
789,2022,AZ,Arizona,Gila,BRFSS,Prevention,Visits to doctor for routine checkup within th...,%,Age-adjusted prevalence,68.8,...,65.0,72.4,"53,922","43,714",4007,PREVENT,CHECKUP,AgeAdjPrv,Annual Checkup,POINT (-110.812096582079 33.7999645115514)
794,2022,AR,Arkansas,Madison,BRFSS,Prevention,Visits to doctor for routine checkup within th...,%,Age-adjusted prevalence,74.5,...,70.2,78.4,"17,486","13,373",5087,PREVENT,CHECKUP,AgeAdjPrv,Annual Checkup,POINT (-93.7242584115005 36.0113363272212)
867,2022,AR,Arkansas,Jackson,BRFSS,Prevention,Visits to doctor for routine checkup within th...,%,Age-adjusted prevalence,76.7,...,72.7,80.3,"16,624","13,317",5067,PREVENT,CHECKUP,AgeAdjPrv,Annual Checkup,POINT (-91.2146324874827 35.5993609746175)


In [16]:
# check year
places_target["Year"].value_counts(dropna=False)

Year
2022    3145
Name: count, dtype: int64

In [17]:
# inspect target values
places_target["Data_Value"].describe()

count    3145.000000
mean       73.823052
std         3.579477
min        59.300000
25%        71.600000
50%        74.400000
75%        76.400000
max        83.100000
Name: Data_Value, dtype: float64

In [18]:
# missingness
print(
    "Missing target values:",
    places_target["Data_Value"].isna().sum()
)

Missing target values: 0


In [19]:
# give target a clear name (for later)
places_target["routine_checkup_pct"] = places_target["Data_Value"]

### 3.2 Standardize County FIPS

County FIPS codes will serve as the common geographic identifier across PLACES, AHRF, HPSA, and SVI. They are standardized to five-character strings so leading zeros are preserved.

In [20]:
# inspect the raw identifier
places_target[
    ["LocationName", "StateAbbr", "LocationID"]
].head(10)

,LocationName,StateAbbr,LocationID
615,Mendocino,CA,6045
736,Chugach,AK,2063
789,Gila,AZ,4007
794,Madison,AR,5087
867,Jackson,AR,5067
908,Polk,AR,5113
1013,Logan,AR,5083
1151,Choctaw,AL,1023
1163,Maricopa,AZ,4013
1217,Fairbanks North Star,AK,2090


In [21]:
# check type
places_target["LocationID"].dtype

dtype('int64')

In [22]:
# convert to string 
places_target["fips"] = (
    pd.to_numeric(
        places_target["LocationID"],
        errors = "coerce"
    )
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

In [23]:
# confirm
places_target[
    ["LocationName", "StateAbbr", "LocationID", "fips"]
].head(10)

,LocationName,StateAbbr,LocationID,fips
615,Mendocino,CA,6045,06045
736,Chugach,AK,2063,02063
789,Gila,AZ,4007,04007
794,Madison,AR,5087,05087
867,Jackson,AR,5067,05067
908,Polk,AR,5113,05113
1013,Logan,AR,5083,05083
1151,Choctaw,AL,1023,01023
1163,Maricopa,AZ,4013,04013
1217,Fairbanks North Star,AK,2090,02090


In [24]:
# validate lengths (counties should have 5)
places_target["fips"].str.len().value_counts(dropna = False)

fips
5    3145
Name: count, dtype: int64

In [25]:
# check for anything that isn't county-like
places_target[
    places_target["fips"].str.len() != 5
][
    ["LocationName", "StateAbbr", "LocationID", "fips"]
]

,LocationName,StateAbbr,LocationID,fips


### 3.3 Check for an aggregate U.S. row

In [26]:
# check for an aggregate U.S. row
places_target[
    places_target["LocationName"]
    .astype(str)
    .str.contains(
        "United States",
        case = False,
        na = False
    )
]

,Year,StateAbbr,StateDesc,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,...,TotalPopulation,TotalPop18plus,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text,Geolocation,routine_checkup_pct,fips


In [27]:
# if there is a national aggregate, we want to exclude it from the county modeling dataset

In [28]:
# verify county uniqueness
print("Rows:", len(places_target))
print("Unique FIPS:", places_target["fips"].nunique())
print(
    "Duplicate FIPS:",
    places_target["fips"].duplicated().sum()
)

Rows: 3145
Unique FIPS: 3145
Duplicate FIPS: 0


### 3.4 Confirm for no extra aggregate rows

In [29]:
print("Final PLACES county rows:", len(places_target))
print(
    "Final unique county FIPS:",
    places_target["fips"].nunique()
)

Final PLACES county rows: 3145
Final unique county FIPS: 3145


---

## 4. CDC/ATSDR Social Vulnerability Index Audit

The 2022 national county-level Social Vulnerability Index provides community vulnerability characteristics primarily for the unsupervised-learning portion of the project.

In [30]:
svi_path = SVI_DIR / "svi_county_2022.csv"

svi = pd.read_csv(
    svi_path,
    low_memory=False
)

print("SVI shape:", svi.shape)

SVI shape: (3144, 158)


In [31]:
svi.head()

,ST,STATE,ST_ABBR,STCNTY,COUNTY,FIPS,LOCATION,AREA_SQMI,E_TOTPOP,M_TOTPOP,...,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
0,1,Alabama,AL,1001,Autauga County,1001,"Autauga County, Alabama",594.454786,58761,0,...,1.1,0.4,0.1,0.1,0.0,0.1,3.3,1.0,0.2,0.3
1,1,Alabama,AL,1003,Baldwin County,1003,"Baldwin County, Alabama",1589.861817,233420,0,...,0.9,0.1,0.2,0.1,0.0,0.1,3.1,0.4,0.4,0.3
2,1,Alabama,AL,1005,Barbour County,1005,"Barbour County, Alabama",885.007619,24877,0,...,0.5,0.1,0.3,0.1,0.0,0.1,1.8,0.7,1.2,0.8
3,1,Alabama,AL,1007,Bibb County,1007,"Bibb County, Alabama",622.469286,22251,0,...,0.3,0.4,0.1,0.1,0.0,0.2,1.7,1.0,0.1,0.1
4,1,Alabama,AL,1009,Blount County,1009,"Blount County, Alabama",644.890376,59077,0,...,0.2,0.2,0.1,0.1,0.2,0.2,2.8,0.7,0.1,0.1


In [32]:
# inspect columns
svi.columns.tolist()

['ST',
 'STATE',
 'ST_ABBR',
 'STCNTY',
 'COUNTY',
 'FIPS',
 'LOCATION',
 'AREA_SQMI',
 'E_TOTPOP',
 'M_TOTPOP',
 'E_HU',
 'M_HU',
 'E_HH',
 'M_HH',
 'E_POV150',
 'M_POV150',
 'E_UNEMP',
 'M_UNEMP',
 'E_HBURD',
 'M_HBURD',
 'E_NOHSDP',
 'M_NOHSDP',
 'E_UNINSUR',
 'M_UNINSUR',
 'E_AGE65',
 'M_AGE65',
 'E_AGE17',
 'M_AGE17',
 'E_DISABL',
 'M_DISABL',
 'E_SNGPNT',
 'M_SNGPNT',
 'E_LIMENG',
 'M_LIMENG',
 'E_MINRTY',
 'M_MINRTY',
 'E_MUNIT',
 'M_MUNIT',
 'E_MOBILE',
 'M_MOBILE',
 'E_CROWD',
 'M_CROWD',
 'E_NOVEH',
 'M_NOVEH',
 'E_GROUPQ',
 'M_GROUPQ',
 'EP_POV150',
 'MP_POV150',
 'EP_UNEMP',
 'MP_UNEMP',
 'EP_HBURD',
 'MP_HBURD',
 'EP_NOHSDP',
 'MP_NOHSDP',
 'EP_UNINSUR',
 'MP_UNINSUR',
 'EP_AGE65',
 'MP_AGE65',
 'EP_AGE17',
 'MP_AGE17',
 'EP_DISABL',
 'MP_DISABL',
 'EP_SNGPNT',
 'MP_SNGPNT',
 'EP_LIMENG',
 'MP_LIMENG',
 'EP_MINRTY',
 'MP_MINRTY',
 'EP_MUNIT',
 'MP_MUNIT',
 'EP_MOBILE',
 'MP_MOBILE',
 'EP_CROWD',
 'MP_CROWD',
 'EP_NOVEH',
 'MP_NOVEH',
 'EP_GROUPQ',
 'MP_GROUPQ',
 'EPL_POV15

In [33]:
# basic audit
print("Rows:", len(svi))
print("Columns:", svi.shape[1])
print("Duplicate rows:", svi.duplicated().sum())

Rows: 3144
Columns: 158
Duplicate rows: 0


In [34]:
# check for missingness
svi.isna().sum().sort_values(ascending = False).head(20)

ST           0
STATE        0
ST_ABBR      0
STCNTY       0
COUNTY       0
FIPS         0
LOCATION     0
AREA_SQMI    0
E_TOTPOP     0
M_TOTPOP     0
E_HU         0
M_HU         0
E_HH         0
M_HH         0
E_POV150     0
M_POV150     0
E_UNEMP      0
M_UNEMP      0
E_HBURD      0
M_HBURD      0
dtype: int64

### 4.1 Standardize SVI FIPS

In [35]:
svi[["FIPS"]].head()

,FIPS
0,1001
1,1003
2,1005
3,1007
4,1009


In [36]:
svi["fips"] = (
    pd.to_numeric(
        svi["FIPS"],
        errors = "coerce"
    )
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

In [37]:
# check
svi[
    ["FIPS", "fips"]
].head()

,FIPS,fips
0,1001,01001
1,1003,01003
2,1005,01005
3,1007,01007
4,1009,01009


In [38]:
# inspect
svi[
    ["STATE", "COUNTY", "FIPS", "fips"]
].head()

,STATE,COUNTY,FIPS,fips
0,Alabama,Autauga County,1001,01001
1,Alabama,Baldwin County,1003,01003
2,Alabama,Barbour County,1005,01005
3,Alabama,Bibb County,1007,01007
4,Alabama,Blount County,1009,01009


In [39]:
# should expect 3,144 unique counties
print("SVI rows:", len(svi))
print("Unique FIPS:", svi["fips"].nunique())
print(
    "Duplicate FIPS:",
    svi["fips"].duplicated().sum()
)

SVI rows: 3144
Unique FIPS: 3144
Duplicate FIPS: 0


### 4.2 Compare Geographic Coverage with PLACES

County FIPS coverage is compared between PLACES and SVI before merging. Any unmatched counties will be documented rather than silently discarded.

In [40]:
# create sets
places_fips = set(places_target["fips"])
svi_fips = set(svi["fips"])

In [41]:
print("PLACES counties:", len(places_fips))
print("SVI counties:", len(svi_fips))

print(
    "PLACES counties not found in SVI:",
    len(places_fips - svi_fips)
)

print(
    "SVI counties not found in PLACES:",
    len(svi_fips - places_fips)
)

PLACES counties: 3145
SVI counties: 3144
PLACES counties not found in SVI: 1
SVI counties not found in PLACES: 0


In [42]:
# inspect unmatched FIPS
places_only = sorted(places_fips - svi_fips)
svi_only = sorted(svi_fips - places_fips)

print("PLACES only:")
print(places_only[:20])

print("\nSVI only:")
print(svi_only[:20])

PLACES only:
['00059']

SVI only:
[]


In [43]:
# flagged because 00 is not a state FIPS prefix
# likely US aggregate
places_target.loc[
    places_target["fips"] == "00059",
    [
        "LocationName",
        "StateAbbr",
        "LocationID",
        "Year",
        "Measure",
        "Data_Value_Type",
        "Data_Value",
        "fips"
    ]
]

,LocationName,StateAbbr,LocationID,Year,Measure,Data_Value_Type,Data_Value,fips
9332,NaN,US,59,2022,Visits to doctor for routine checkup within th...,Age-adjusted prevalence,74.2,00059


In [44]:
# remove national aggregate record; retain county-level observations only
places_target = places_target[
    places_target["StateAbbr"] != "US"
].copy()

In [45]:
places_fips = set(places_target["fips"])
svi_fips = set(svi["fips"])

print("PLACES counties:", len(places_fips))
print("SVI counties:", len(svi_fips))

print(
    "PLACES counties not found in SVI:",
    len(places_fips - svi_fips)
)

print(
    "SVI counties not found in PLACES:",
    len(svi_fips - places_fips)
)

PLACES counties: 3144
SVI counties: 3144
PLACES counties not found in SVI: 0
SVI counties not found in PLACES: 0


In [46]:
# verify uniqueness
print("Unique PLACES FIPS:", places_target["fips"].nunique())
print("Duplicate PLACES FIPS:", places_target["fips"].duplicated().sum())

Unique PLACES FIPS: 3144
Duplicate PLACES FIPS: 0


### PLACES Geographic Coverage Note

The initial age-adjusted routine-checkup subset contained 3,145 records. One record had `StateAbbr = "US"` and represented a national aggregate rather than a county-level observation. Its numeric `LocationID` of 59 became `00059` when county identifiers were zero-padded to five characters.

Because the project analyzes U.S. counties and county equivalents, this national aggregate was excluded.

After exclusion, PLACES contained 3,144 unique county/county-equivalent FIPS codes, matching the 3,144 counties in the 2022 SVI dataset.

### 4.3 Test Merge SVI and PLACES 

In [47]:
# test merge
svi_test = svi[
    ["fips", "STATE", "COUNTY"]
].copy()

places_svi_test = places_target.merge(
    svi_test,
    on="fips",
    how="left",
    validate="one_to_one"
)

print("PLACES rows before merge:", len(places_target))
print("Rows after SVI merge:", len(places_svi_test))

PLACES rows before merge: 3144
Rows after SVI merge: 3144


In [48]:
places_svi_test[
    ["fips", "LocationName", "STATE", "COUNTY"]
].head(10)

,fips,LocationName,STATE,COUNTY
0,06045,Mendocino,California,Mendocino County
1,02063,Chugach,Alaska,Chugach Census Area
2,04007,Gila,Arizona,Gila County
3,05087,Madison,Arkansas,Madison County
4,05067,Jackson,Arkansas,Jackson County
5,05113,Polk,Arkansas,Polk County
6,05083,Logan,Arkansas,Logan County
7,01023,Choctaw,Alabama,Choctaw County
8,04013,Maricopa,Arizona,Maricopa County
9,02090,Fairbanks North Star,Alaska,Fairbanks North Star Borough


In [49]:
print("Missing STATE after merge:", places_svi_test["STATE"].isna().sum())
print("Missing COUNTY after merge:", places_svi_test["COUNTY"].isna().sum())

Missing STATE after merge: 0
Missing COUNTY after merge: 0


### PLACES–SVI Geographic Alignment Result

After removing the national PLACES aggregate, both PLACES and SVI contained 3,144 unique U.S. county/county-equivalent FIPS codes.

A one-to-one merge on standardized five-character FIPS codes preserved all 3,144 PLACES observations with no unmatched SVI counties.

**Decision:** PLACES and SVI are geographically compatible for the county-level analysis.

---

## 5. HRSA AHRF Audit

The Area Health Resources Files (AHRF) provide county-level healthcare workforce, infrastructure, population, and geographic characteristics.

Because the full AHRF file contains thousands of variables and includes current, historical, and non-project geographic records, this section:

1. identifies the main county-level AHRF file,
2. validates the candidate variables selected for this project,
3. standardizes county FIPS codes,
4. compares AHRF geographic coverage with the PLACES county universe, and
5. documents missingness in the proposed predictor variables.

No imputation or feature engineering is performed at this stage.

### 5.1 Load Data

The full AHRF dataset is stored as a ZIP archive. Rather than extracting or loading all variables immediately, the archive is inspected first to identify the main county-level CSV.

In [50]:
# define AHRF paths
import zipfile

ahrf_zip_path = AHRF_DIR / "ahrf_county_2023_2024.zip"
ahrf_candidate_path = (
    REFERENCE_DIR / "ahrf_candidate_variables_team17.csv"
)

In [51]:
# inspect the ZIP contents
with zipfile.ZipFile(ahrf_zip_path, "r") as z:
    ahrf_zip_files = z.namelist()

for file in ahrf_zip_files:
    print(file)

AHRF 2023-2024 CSV/2023-2024_AHRFDUA.docx
AHRF 2023-2024 CSV/ahrf2024_Feb2025.csv
AHRF 2023-2024 CSV/CSV Files by Categories/
AHRF 2023-2024 CSV/CSV Files by Categories/ahrf2024env.csv
AHRF 2023-2024 CSV/CSV Files by Categories/ahrf2024exp.csv
AHRF 2023-2024 CSV/CSV Files by Categories/ahrf2024geo_Feb2025.csv
AHRF 2023-2024 CSV/CSV Files by Categories/ahrf2024hf.csv
AHRF 2023-2024 CSV/CSV Files by Categories/ahrf2024hp.csv
AHRF 2023-2024 CSV/CSV Files by Categories/ahrf2024pop.csv
AHRF 2023-2024 CSV/CSV Files by Categories/ahrf2024utl.csv


In [52]:
# we want the main combined AHRF file

In [53]:
# identify the main CSV (w/o hardcoding)
with zipfile.ZipFile(ahrf_zip_path, "r") as z:
    csv_files = [
        file
        for file in z.namelist()
        if file.lower().endswith(".csv")
    ]

main_csv_candidates = [
    file
    for file in csv_files
    if "CSV Files by Categories" not in file
]

main_csv_candidates

['AHRF 2023-2024 CSV/ahrf2024_Feb2025.csv']

In [54]:
ahrf_main_csv = main_csv_candidates[0]

print("Main AHRF file:")
print(ahrf_main_csv)

Main AHRF file:
AHRF 2023-2024 CSV/ahrf2024_Feb2025.csv


In [55]:
# read header, it will tell us 4300
with zipfile.ZipFile(ahrf_zip_path, "r") as z:
    with z.open(ahrf_main_csv) as f:
        ahrf_header = pd.read_csv(
            f,
            nrows=0
        )

print("Number of AHRF columns:", len(ahrf_header.columns))

Number of AHRF columns: 4300


### 5.2 Validate Candidate Variables

AHRF contains approximately 4,300 variables in this release. The project therefore uses a curated candidate-variable reference file rather than examining every AHRF field.

Candidate variables were selected based on the project's conceptual domains:

- provider supply,
- healthcare infrastructure,
- geographic/access context,
- and selected community barriers.

This section verifies that each proposed variable exists in the downloaded AHRF release before any modeling or feature engineering is performed.

In [56]:
# load candidate-variable reference file
ahrf_candidates = pd.read_csv(
    ahrf_candidate_path
)

ahrf_candidates

,variable,year,category,description,recommended_use,transformation,priority,notes
0,fips_st_cnty,NaN,Geography,FIPS state + county code,Join key,Convert to 5-character zero-padded string,Required,Primary key for joining to PLACES/SVI/HPSA
1,popn_est_22,2022.0,Population,Population estimate,Denominator,Use to construct provider/facility rates,Required,Prefer rates over raw counts for county compar...
2,phys_nf_prim_care_pc_exc_rsdt_22,2022.0,Provider supply,Non-federal primary care physicians in patient...,Core supervised predictor,"Per 100,000 population",High,Aggregate primary care measure; includes gener...
3,np_npi_22,2022.0,Provider supply,Nurse practitioners with NPI,Core supervised predictor,"Per 100,000 population",High,Captures nonphysician primary-care capacity
4,pa_npi_22,2022.0,Provider supply,Physician assistants with NPI,Core supervised predictor,"Per 100,000 population",High,Captures nonphysician outpatient-care capacity
5,hosp_22,2022.0,Healthcare infrastructure,Total number of hospitals,Candidate supervised/unsupervised predictor,"Per 100,000 population",Medium,Raw count is strongly population-dependent
6,hosp_beds_22,2022.0,Healthcare infrastructure,"Hospital beds, total hospitals",Candidate supervised/unsupervised predictor,"Beds per 1,000 population",Medium,More interpretable as a population-normalized ...
7,fedly_qualfd_hlth_ctr_22,2022.0,Healthcare infrastructure,Number of Federally Qualified Health Centers,Candidate supervised/unsupervised predictor,"Per 100,000 population",High,Potentially relevant to safety-net primary-car...
8,rural_hlth_clincs_22,2022.0,Healthcare infrastructure,Number of Rural Health Clinics,Candidate predictor,"Per 100,000 population",Medium,May be especially informative in nonmetro coun...
9,rural_urban_contnm_23,2023.0,Geographic/access context,Rural-Urban Continuum Code,Structural access predictor / stratification v...,Treat as categorical or collapse to metro/nonm...,High,Useful for grouped residual analysis and spati...


In [57]:
# get the proposed variable names
candidate_vars = (
    ahrf_candidates["variable"]
    .dropna()
    .tolist()
)

candidate_vars

['fips_st_cnty',
 'popn_est_22',
 'phys_nf_prim_care_pc_exc_rsdt_22',
 'np_npi_22',
 'pa_npi_22',
 'hosp_22',
 'hosp_beds_22',
 'fedly_qualfd_hlth_ctr_22',
 'rural_hlth_clincs_22',
 'rural_urban_contnm_23',
 'popn_densty_per_squr_mi_20',
 'pers_noins_lt65_pct_21']

In [58]:
# check whether every candidate exists
ahrf_columns = set(ahrf_header.columns)

candidate_validation = pd.DataFrame({
    "variable": candidate_vars,
    "found_in_ahrf": [
        var in ahrf_columns
        for var in candidate_vars
    ]
})

candidate_validation

,variable,found_in_ahrf
0,fips_st_cnty,True
1,popn_est_22,True
2,phys_nf_prim_care_pc_exc_rsdt_22,True
3,np_npi_22,True
4,pa_npi_22,True
5,hosp_22,True
6,hosp_beds_22,True
7,fedly_qualfd_hlth_ctr_22,True
8,rural_hlth_clincs_22,True
9,rural_urban_contnm_23,True


In [59]:
# summarize
# if all of our currently proposed variables are present
# second output should be an empty table
print(
    candidate_validation["found_in_ahrf"]
    .value_counts()
)

print("\nMissing proposed variables:")

candidate_validation[
    ~candidate_validation["found_in_ahrf"]
]

found_in_ahrf
True    12
Name: count, dtype: int64

Missing proposed variables:


,variable,found_in_ahrf


In [60]:
# load only needed fields
ahrf_id_cols = [
    "fips_st_cnty",
    "st_name",
    "st_name_abbrev",
    "cnty_name",
]

In [61]:
# avoid duplicates (`fips_st_cnty` is already in the candidate list)
ahrf_usecols = list(
    dict.fromkeys(
        ahrf_id_cols + candidate_vars
    )
)

ahrf_usecols

['fips_st_cnty',
 'st_name',
 'st_name_abbrev',
 'cnty_name',
 'popn_est_22',
 'phys_nf_prim_care_pc_exc_rsdt_22',
 'np_npi_22',
 'pa_npi_22',
 'hosp_22',
 'hosp_beds_22',
 'fedly_qualfd_hlth_ctr_22',
 'rural_hlth_clincs_22',
 'rural_urban_contnm_23',
 'popn_densty_per_squr_mi_20',
 'pers_noins_lt65_pct_21']

In [62]:
# load only those columns
with zipfile.ZipFile(ahrf_zip_path, "r") as z:
    with z.open(ahrf_main_csv) as f:
        ahrf = pd.read_csv(
            f,
            usecols = ahrf_usecols,
            low_memory = False
        )

print("Selected AHRF shape:", ahrf.shape)

Selected AHRF shape: (3240, 15)


In [63]:
ahrf.head()

,fips_st_cnty,st_name,st_name_abbrev,cnty_name,rural_urban_contnm_23,phys_nf_prim_care_pc_exc_rsdt_22,pa_npi_22,np_npi_22,hosp_22,hosp_beds_22,rural_hlth_clincs_22,fedly_qualfd_hlth_ctr_22,popn_est_22,pers_noins_lt65_pct_21,popn_densty_per_squr_mi_20
0,1001,Alabama,AL,Autauga,2.0,25.0,5.0,28.0,1.0,77.0,0.0,2.0,59759.0,10.0,98.9
1,1003,Alabama,AL,Baldwin,3.0,160.0,41.0,195.0,4.0,418.0,0.0,4.0,246435.0,11.0,145.8
2,1005,Alabama,AL,Barbour,6.0,8.0,1.0,17.0,1.0,47.0,2.0,3.0,24706.0,12.7,28.5
3,1007,Alabama,AL,Bibb,1.0,21.0,5.0,31.0,1.0,166.0,1.0,8.0,22005.0,11.4,35.8
4,1009,Alabama,AL,Blount,1.0,12.0,0.0,20.0,1.0,25.0,6.0,1.0,59512.0,12.8,91.7


In [64]:
ahrf.dtypes

fips_st_cnty                          int64
st_name                              object
st_name_abbrev                       object
cnty_name                            object
rural_urban_contnm_23               float64
phys_nf_prim_care_pc_exc_rsdt_22    float64
pa_npi_22                           float64
np_npi_22                           float64
hosp_22                             float64
hosp_beds_22                        float64
rural_hlth_clincs_22                float64
fedly_qualfd_hlth_ctr_22            float64
popn_est_22                         float64
pers_noins_lt65_pct_21              float64
popn_densty_per_squr_mi_20          float64
dtype: object

### 5.3 Standardize County FIPS

AHRF county identifiers are standardized to five-character FIPS strings so they can be compared with the PLACES and SVI county universe.

AHRF contains more geographic records than PLACES, including territories and some historical county-equivalent records. These additional records are documented before restricting AHRF to the project's PLACES-defined county universe.

In [65]:
# standardize FIPS (already created helper)
def standardize_fips(series):
    """
    Convert county FIPS identifiers to
    five-character strings.
    """
    return (
        pd.to_numeric(
            series,
            errors="coerce"
        )
        .astype("Int64")
        .astype(str)
        .str.zfill(5)
    )

In [66]:
# use it
ahrf["fips"] = standardize_fips(
    ahrf["fips_st_cnty"]
)

In [67]:
# inspect
# expected: 1001 → 01001 Alabama
ahrf[
    [
        "fips_st_cnty",
        "fips",
        "st_name",
        "cnty_name"
    ]
].head(10)

,fips_st_cnty,fips,st_name,cnty_name
0,1001,01001,Alabama,Autauga
1,1003,01003,Alabama,Baldwin
2,1005,01005,Alabama,Barbour
3,1007,01007,Alabama,Bibb
4,1009,01009,Alabama,Blount
5,1011,01011,Alabama,Bullock
6,1013,01013,Alabama,Butler
7,1015,01015,Alabama,Calhoun
8,1017,01017,Alabama,Chambers
9,1019,01019,Alabama,Cherokee


In [68]:
# validate the FIPS fields
print("AHRF rows:", len(ahrf))

print(
    "Unique AHRF FIPS:",
    ahrf["fips"].nunique()
)

print(
    "Duplicate AHRF FIPS:",
    ahrf["fips"].duplicated().sum()
)

print("\nFIPS lengths:")

print(
    ahrf["fips"]
    .str.len()
    .value_counts()
)

AHRF rows: 3240
Unique AHRF FIPS: 3240
Duplicate AHRF FIPS: 0

FIPS lengths:
fips
5    3240
Name: count, dtype: int64


In [69]:
ahrf_fips = set(ahrf["fips"])

print("PLACES counties:", len(places_fips))
print("AHRF geographic records:", len(ahrf_fips))

print(
    "PLACES not found in AHRF:",
    len(places_fips - ahrf_fips)
)

print(
    "AHRF not found in PLACES:",
    len(ahrf_fips - places_fips)
)

PLACES counties: 3144
AHRF geographic records: 3240
PLACES not found in AHRF: 0
AHRF not found in PLACES: 96


In [70]:
# AHRF contains every county in the PLACES universe, plus 96 additional geographic/historical records

In [71]:
# inspect those extras
ahrf_only_fips = sorted(
    ahrf_fips - places_fips
)

ahrf_only = ahrf[
    ahrf["fips"].isin(ahrf_only_fips)
][
    ["fips", "st_name", "cnty_name"]
].copy()

ahrf_only.head(20)

,fips,st_name,cnty_name
91,02201,Alaska,Pr of Wales-OutrKetch(CA)
94,02232,Alaska,Skagway-Hoonah-Angoon(CA)
96,02261,Alaska,Valdez-Cordova (CA)
98,02280,Alaska,Wrangell-Petersburg (CA)
313,09001,Connecticut,Fairfield
314,09003,Connecticut,Hartford
315,09005,Connecticut,Litchfield
316,09007,Connecticut,Middlesex
317,09009,Connecticut,New Haven
318,09011,Connecticut,New London


In [72]:
# summarize
ahrf_only["st_name"].value_counts()

st_name
Puerto Rico          78
Connecticut           8
Alaska                4
US Virgin Islands     3
Virginia              2
Guam                  1
Name: count, dtype: int64

In [73]:
# PLACES is defining a different current county universe
# So, we should not call these missing or bad records
# they are simply outside the target geography for our project

In [74]:
# after taking note of these locations
# restrict AHRF to the PLACES county universe
ahrf_county = ahrf[
    ahrf["fips"].isin(
        places_fips
    )
].copy()

In [75]:
print(
    "AHRF rows after geographic alignment:",
    len(ahrf_county)
)

print(
    "Unique county FIPS:",
    ahrf_county["fips"].nunique()
)

AHRF rows after geographic alignment: 3144
Unique county FIPS: 3144


In [76]:
# Now our three datasets have the same county universe:
# PLACES    3,144
# SVI       3,144
# AHRF      3,144

### 5.4 Inspect Missingness and Coverage

Matching county FIPS does not guarantee that every AHRF variable is populated for every county. Missingness is therefore evaluated after aligning AHRF to the 3,144-county PLACES universe.

No counties or variables are removed at this stage. The purpose is to identify patterns that will inform later cleaning and feature-engineering decisions.

#### This is where we discover the Connecticut problem our peer reviewers predicted.

In [77]:
# define actual candidate feature fields
# we don't need the FIPS field in missingness summary

ahrf_feature_cols = [
    var
    for var in candidate_vars
    if var != "fips_st_cnty"
]

In [78]:
# calcualte missing values and percentages

ahrf_missing = pd.DataFrame({
    "missing_n":
        ahrf_county[
            ahrf_feature_cols
        ].isna().sum(),

    "missing_pct":
        (
            ahrf_county[
                ahrf_feature_cols
            ]
            .isna()
            .mean()
            .mul(100)
        )
})

ahrf_missing = (
    ahrf_missing
    .sort_values(
        "missing_n",
        ascending = False
    )
)

ahrf_missing

,missing_n,missing_pct
fedly_qualfd_hlth_ctr_22,13,0.413486
rural_hlth_clincs_22,13,0.413486
pers_noins_lt65_pct_21,10,0.318066
hosp_beds_22,9,0.286260
np_npi_22,9,0.286260
phys_nf_prim_care_pc_exc_rsdt_22,9,0.286260
pa_npi_22,9,0.286260
popn_densty_per_squr_mi_20,9,0.286260
hosp_22,9,0.286260
popn_est_22,0,0.000000


In [79]:
# readability
ahrf_missing["missing_pct"] = (
    ahrf_missing[
        "missing_pct"
    ].round(2)
)

ahrf_missing

,missing_n,missing_pct
fedly_qualfd_hlth_ctr_22,13,0.41
rural_hlth_clincs_22,13,0.41
pers_noins_lt65_pct_21,10,0.32
hosp_beds_22,9,0.29
np_npi_22,9,0.29
phys_nf_prim_care_pc_exc_rsdt_22,9,0.29
pa_npi_22,9,0.29
popn_densty_per_squr_mi_20,9,0.29
hosp_22,9,0.29
popn_est_22,0,0.00


In [80]:
# at first glance, overall missingness is low
# BUT we need to care more about where it occurs 

In [81]:
# find counties with missing AHRF data
missing_mask = (
    ahrf_county[
        ahrf_feature_cols
    ]
    .isna()
    .any(axis = 1)
)

ahrf_missing_counties = (
    ahrf_county.loc[
        missing_mask,
        [
            "fips",
            "st_name",
            "cnty_name"
        ]
        + ahrf_feature_cols
    ]
)

print(
    "Counties with at least one "
    "missing candidate AHRF value:",
    len(ahrf_missing_counties)
)

Counties with at least one missing candidate AHRF value: 14


In [82]:
# inspect:
ahrf_missing_counties[
    [
        "fips",
        "st_name",
        "cnty_name"
    ]
]

,fips,st_name,cnty_name
72,02063,Alaska,Chugach (CA)
73,02066,Alaska,Copper River (CA)
89,02195,Alaska,Petersburg (B)
97,02275,Alaska,Wrangell (B)
321,09110,Connecticut,Capitol PR
322,09120,Connecticut,Greater Bridgeport PR
323,09130,Connecticut,Lower CT River Valley PR
324,09140,Connecticut,Naugatuck Valley PR
325,09150,Connecticut,Northeastern CT PR
326,09160,Connecticut,Northwest Hills PR


In [83]:
# let's investigate missingness by state 
ahrf_missing_counties[
    "st_name"
].value_counts()

st_name
Connecticut    9
Alaska         4
Hawaii         1
Name: count, dtype: int64

In [84]:
# Connecticut appears to be the main issue
# inspect
ahrf_missing_counties[
    ahrf_missing_counties[
        "st_name"
    ] == "Connecticut"
][
    [
        "fips",
        "cnty_name"
    ]
]

# we should see the 9 Connecticut planning regions after we run this cell

,fips,cnty_name
321,09110,Capitol PR
322,09120,Greater Bridgeport PR
323,09130,Lower CT River Valley PR
324,09140,Naugatuck Valley PR
325,09150,Northeastern CT PR
326,09160,Northwest Hills PR
327,09170,South Central CT PR
328,09180,Southeastern CT PR
329,09190,Western CT PR


#### Important audit finding: 

These are the new Connecticut county equivalents used in PLACES/SVI.
AHRF includes these geographic records, but many 2022 healthcare-resource fields are blank for them because the underlying 2022 source data were still based on Connecticut's former eight-county geography.

In [85]:
# inspect FQHC and Rural Clinic missingness 
# FQHC:
ahrf_county.loc[
    ahrf_county[
        "fedly_qualfd_hlth_ctr_22"
    ].isna(),
    [
        "fips",
        "st_name",
        "cnty_name"
    ]
]

,fips,st_name,cnty_name
72,02063,Alaska,Chugach (CA)
73,02066,Alaska,Copper River (CA)
89,02195,Alaska,Petersburg (B)
97,02275,Alaska,Wrangell (B)
321,09110,Connecticut,Capitol PR
322,09120,Connecticut,Greater Bridgeport PR
323,09130,Connecticut,Lower CT River Valley PR
324,09140,Connecticut,Naugatuck Valley PR
325,09150,Connecticut,Northeastern CT PR
326,09160,Connecticut,Northwest Hills PR


In [86]:
# Similarly, Rural Clinic:

ahrf_county.loc[
    ahrf_county[
        "rural_hlth_clincs_22"
    ].isna(),
    [
        "fips",
        "st_name",
        "cnty_name"
    ]
]

,fips,st_name,cnty_name
72,02063,Alaska,Chugach (CA)
73,02066,Alaska,Copper River (CA)
89,02195,Alaska,Petersburg (B)
97,02275,Alaska,Wrangell (B)
321,09110,Connecticut,Capitol PR
322,09120,Connecticut,Greater Bridgeport PR
323,09130,Connecticut,Lower CT River Valley PR
324,09140,Connecticut,Naugatuck Valley PR
325,09150,Connecticut,Northeastern CT PR
326,09160,Connecticut,Northwest Hills PR


In [87]:
# check uninsured missingness
ahrf_county.loc[
    ahrf_county[
        "pers_noins_lt65_pct_21"
    ].isna(),
    [
        "fips",
        "st_name",
        "cnty_name"
    ]
]

,fips,st_name,cnty_name
321,09110,Connecticut,Capitol PR
322,09120,Connecticut,Greater Bridgeport PR
323,09130,Connecticut,Lower CT River Valley PR
324,09140,Connecticut,Naugatuck Valley PR
325,09150,Connecticut,Northeastern CT PR
326,09160,Connecticut,Northwest Hills PR
327,09170,Connecticut,South Central CT PR
328,09180,Connecticut,Southeastern CT PR
329,09190,Connecticut,Western CT PR
562,15005,Hawaii,Kalawao


#### Important distinction:

At this point, we know:

##### Geographic coverage

- PLACES → 3,144
- SVI    → 3,144
- AHRF   → 3,144

which means that, 

**AHRF has an actual row for every PLACES county.**

But:

**Some AHRF variables are missing within those rows.**

I think it is important to note that these are two different concepts:

**unmatched geography vs. matched county with missing feature data**

##### This is an important distinction that I would like to make early in this audit notebook.

In [88]:
# test AHRF merge
# one-on-one test merge: PLACES identifiers and AHRF (for now)

places_ahrf_test = (
    places_target[
        [
            "fips",
            "LocationName",
            "StateAbbr",
            "routine_checkup_pct"
        ]
    ]
    .merge(
        ahrf_county,
        on = "fips",
        how = "left",
        validate = "one_to_one",
        indicator = True
    )
)

In [89]:
print(
    "PLACES rows before merge:",
    len(places_target)
)

print(
    "Rows after AHRF merge:",
    len(places_ahrf_test)
)

print("\nMerge status:")

print(
    places_ahrf_test[
        "_merge"
    ].value_counts()
)

PLACES rows before merge: 3144
Rows after AHRF merge: 3144

Merge status:
_merge
both          3144
left_only        0
right_only       0
Name: count, dtype: int64


#### Note:
##### The geography matches perfectly even though some predictor values are missing.

### AHRF Audit Result

The 2023–2024 AHRF release contains 3,240 geographic records and approximately 4,300 variables. All candidate variables identified for the initial project feature set were present in the downloaded release.

After standardizing county FIPS codes, all 3,144 PLACES counties/county equivalents were represented in AHRF. AHRF contained 96 additional geographic records outside the PLACES target universe, including Puerto Rico and other U.S. territories as well as several historical county-equivalent records. These records were documented and excluded from the project-aligned AHRF subset.

After restricting AHRF to the PLACES county universe, the dataset contained one record for each of the 3,144 project counties.

Candidate-variable missingness was generally low but geographically structured. Most 2022 healthcare-resource variables were unavailable for the nine current Connecticut planning regions because of differences between historical and current Connecticut county geography. Selected FQHC and Rural Health Clinic variables also contained missing values for several Alaska county equivalents, while the 2021 uninsured measure was additionally unavailable for Kalawao County, Hawaii.

**Decision:** AHRF is geographically compatible with the PLACES county universe. Missing predictor values will be retained and documented at this stage; decisions regarding imputation, feature exclusion, or alternative geographic handling will be made during the cleaning/feature-engineering stage rather than during the data audit.

---

## 6. HRSA HPSA Audit

The Health Professional Shortage Area (HPSA) files provide shortage designations for Primary Care, Mental Health, and Dental Health.

Unlike PLACES, SVI, and AHRF, the raw HPSA files are not one-row-per-county datasets. A single HPSA designation may span multiple counties, and one county may be associated with multiple HPSA designations.

This section:

1. loads and audits each HPSA discipline,
2. standardizes county FIPS codes,
3. identifies designation types relevant to geographic/population shortage,
4. reconstructs which designations appear to have been active at the end of 2022,
5. deduplicates repeated HPSA-county combinations, and
6. aggregates qualifying designations to one row per county.

Facility-based HPSAs are retained in the raw data but excluded from the primary county-level shortage representation because they describe specific facilities rather than general county/population shortage conditions.

In [90]:
HPSA_PC_PATH = HPSA_DIR / "hpsa_primary_care_2026-09-25.csv"
HPSA_MH_PATH = HPSA_DIR / "hpsa_mental_health_2026-09-25.csv"
HPSA_DH_PATH = HPSA_DIR / "hpsa_dental_health_2026-09-25.csv"

### clean HPSA county FIPS

HPSA files contain values like:

- 06037
- 72075
- XXXXX

So don't use the generic numeric FIPS function here, because XXXXX is a legitimate raw placeholder and should become missing rather than being coerced strangely.

In [91]:
# use:

def clean_hpsa_fips(series):
    """
    Retain valid five-digit county FIPS codes.
    Invalid/non-county geographic identifiers become missing.
    """
    s = (
        series
        .astype("string")
        .str.strip()
    )

    return s.where(
        s.str.fullmatch(r"\d{5}")
    )

### 6.1 Primary Care

The Primary Care HPSA file is audited first to understand its structure, designation types, geographic identifiers, dates, and current status fields.

In [92]:
# load:

hpsa_pc = pd.read_csv(
    HPSA_PC_PATH,
    low_memory = False
)

print("Primary Care HPSA shape:", hpsa_pc.shape)

Primary Care HPSA shape: (80199, 66)


In [93]:
# inspect important fields

hpsa_pc[
    [
        "HPSA ID",
        "Designation Type",
        "HPSA Discipline Class",
        "HPSA Score",
        "HPSA Status",
        "HPSA Designation Date",
        "Withdrawn Date",
        "State and County Federal Information Processing Standard Code",
        "Common County Name"
    ]
].head()

,HPSA ID,Designation Type,HPSA Discipline Class,HPSA Score,HPSA Status,HPSA Designation Date,Withdrawn Date,State and County Federal Information Processing Standard Code,Common County Name
0,1689996802,Federally Qualified Health Center,Primary Care,24,Designated,10/26/2002,NaN,XXXXX,Not Determined
1,1688361205,High Needs Geographic HPSA,Primary Care,25,Designated,06/01/2012,NaN,68410,"Utrik, MH"
2,1688361205,High Needs Geographic HPSA,Primary Care,25,Designated,06/01/2012,NaN,68360,"Rongrik, MH"
3,1688361205,High Needs Geographic HPSA,Primary Care,25,Designated,06/01/2012,NaN,68007,"Ailinginae, MH"
4,1688361205,High Needs Geographic HPSA,Primary Care,25,Designated,06/01/2012,NaN,68385,"Toke, MH"


In [94]:
# confirm discipline

hpsa_pc[
    "HPSA Discipline Class"
].value_counts(dropna = False)

HPSA Discipline Class
Primary Care    80199
Name: count, dtype: int64

In [95]:
# inspect current status

hpsa_pc[
    "HPSA Status"
].value_counts(dropna = False)

HPSA Status
Withdrawn                  48840
Designated                 21444
Proposed For Withdrawal     9915
Name: count, dtype: int64

### current status is not enough to determine whether a designation was active in 2022

In [96]:
# inspect designation types

hpsa_pc[
    "Designation Type"
].value_counts()

Designation Type
HPSA Population                                                                41229
Geographic HPSA                                                                26196
High Needs Geographic HPSA                                                      4809
Rural Health Clinic                                                             3472
Federally Qualified Health Center                                               1584
Indian Health Service, Tribal Health, and Urban Indian Health Organizations     1263
Correctional Facility                                                           1051
Federally Qualified Health Center Look A Like                                    342
Other Facility                                                                   253
Name: count, dtype: int64

In [97]:
# the three types we currently want for the county shortage representation are:

HPSA_DESIGNATION_TYPES = [
    "Geographic HPSA",
    "High Needs Geographic HPSA",
    "HPSA Population"
]

### Why?

Because they characterize **geographic or population shortage**, rather than a shortage designation tied specifically to one facility.

In [98]:
# standardize Primary Care FIPS

hpsa_pc["fips"] = clean_hpsa_fips(
    hpsa_pc[
        "State and County Federal Information Processing Standard Code"
    ]
)

In [99]:
# check

print(
    "Valid county FIPS:",
    hpsa_pc["fips"].notna().sum()
)

print(
    "Invalid/missing county FIPS:",
    hpsa_pc["fips"].isna().sum()
)

print(
    "Unique valid county FIPS:",
    hpsa_pc["fips"].nunique()
)

Valid county FIPS: 79276
Invalid/missing county FIPS: 923
Unique valid county FIPS: 3224


### Keep in mind: 

That includes territories and other geographies outside the PLACES universe.

### 6.2 Mental Health

The Mental Health HPSA file is audited using the same structure as Primary Care so the three disciplines can later be processed consistently.

In [100]:
# load

hpsa_mh = pd.read_csv(
    HPSA_MH_PATH,
    low_memory = False
)

print("Mental Health HPSA shape:", hpsa_mh.shape)

Mental Health HPSA shape: (41250, 65)


In [101]:
# standardize FIPS

hpsa_mh["fips"] = clean_hpsa_fips(
    hpsa_mh[
        "State and County Federal Information Processing Standard Code"
    ]
)

In [102]:
# audit:

print(
    "Valid county FIPS:",
    hpsa_mh["fips"].notna().sum()
)

print(
    "Invalid/missing county FIPS:",
    hpsa_mh["fips"].isna().sum()
)

print(
    "Unique valid county FIPS:",
    hpsa_mh["fips"].nunique()
)

Valid county FIPS: 40848
Invalid/missing county FIPS: 402
Unique valid county FIPS: 3266


In [103]:
# inspect

hpsa_mh["HPSA Status"].value_counts()

HPSA Status
Withdrawn                  18633
Designated                 14367
Proposed For Withdrawal     8250
Name: count, dtype: int64

In [104]:
# and

hpsa_mh["Designation Type"].value_counts()

Designation Type
HPSA Population                                                                19261
Geographic HPSA                                                                 8106
High Needs Geographic HPSA                                                      6102
Rural Health Clinic                                                             3269
Federally Qualified Health Center                                               1571
Indian Health Service, Tribal Health, and Urban Indian Health Organizations     1225
Correctional Facility                                                           1026
Federally Qualified Health Center Look A Like                                    304
State Mental Hospital                                                            212
Other Facility                                                                   174
Name: count, dtype: int64

#### You should see the same basic designation structure as Primary Care.

### 6.3 Dental Health

The Dental Health HPSA file is audited using the same procedures as Primary Care and Mental Health.

In [105]:
# load

hpsa_dh = pd.read_csv(
    HPSA_DH_PATH,
    low_memory = False
)

print("Dental Health HPSA shape:", hpsa_dh.shape)

Dental Health HPSA shape: (47436, 65)


In [106]:
# standardize FIPS

hpsa_dh["fips"] = clean_hpsa_fips(
    hpsa_dh[
        "State and County Federal Information Processing Standard Code"
    ]
)

In [107]:
# audit

print(
    "Valid county FIPS:",
    hpsa_dh["fips"].notna().sum()
)

print(
    "Invalid/missing county FIPS:",
    hpsa_dh["fips"].isna().sum()
)

print(
    "Unique valid county FIPS:",
    hpsa_dh["fips"].nunique()
)

Valid county FIPS: 47165
Invalid/missing county FIPS: 271
Unique valid county FIPS: 3136


In [108]:
# also inspect

hpsa_dh["HPSA Status"].value_counts()

HPSA Status
Withdrawn                  23282
Designated                 16877
Proposed For Withdrawal     7277
Name: count, dtype: int64

In [109]:
# and

hpsa_dh["Designation Type"].value_counts()

Designation Type
HPSA Population                                                                31915
Geographic HPSA                                                                 6176
Rural Health Clinic                                                             3281
High Needs Geographic HPSA                                                      2109
Federally Qualified Health Center                                               1564
Indian Health Service, Tribal Health, and Urban Indian Health Organizations     1208
Correctional Facility                                                            818
Federally Qualified Health Center Look A Like                                    299
Other Facility                                                                    66
Name: count, dtype: int64

### 6.4 Reconstruct 2022 Designations

The downloaded HPSA files are current extracts rather than official annual 2022 snapshots.

To approximate shortage conditions relevant to the 2022 PLACES outcome, designation dates and withdrawal dates are used to reconstruct whether each geographic/population HPSA appears to have been active on December 31, 2022.

A designation is treated as active at the 2022 year-end snapshot when:

- its designation date is on or before December 31, 2022, and
- its withdrawal date is either missing or after December 31, 2022.

Only Geographic HPSA, High Needs Geographic HPSA, and HPSA Population designations are included in the primary county-level shortage representation.

**This is an approximate historical reconstruction from a current HRSA extract and should not be interpreted as an official HRSA 2022 snapshot.**

In [110]:
# parse dates consistently
# create a helper:

HPSA_CUTOFF = pd.Timestamp("2022-12-31")

def prepare_hpsa_2022(df):
    """
    Prepare a current HPSA extract for approximate
    2022 year-end designation reconstruction.
    """

    out = df.copy()

    out["designation_date"] = pd.to_datetime(
        out["HPSA Designation Date"],
        errors="coerce"
    )

    out["withdrawn_date"] = pd.to_datetime(
        out["Withdrawn Date"],
        errors="coerce"
    )

    out["active_2022_eoy"] = (
        out["designation_date"].notna()
        &
        (out["designation_date"] <= HPSA_CUTOFF)
        &
        (
            out["withdrawn_date"].isna()
            |
            (out["withdrawn_date"] > HPSA_CUTOFF)
        )
    )

    out["eligible_designation_type"] = (
        out["Designation Type"]
        .isin(HPSA_DESIGNATION_TYPES)
    )

    return out

In [111]:
# let's run them: 

hpsa_pc_2022 = prepare_hpsa_2022(hpsa_pc)
hpsa_mh_2022 = prepare_hpsa_2022(hpsa_mh)
hpsa_dh_2022 = prepare_hpsa_2022(hpsa_dh)

In [112]:
# check date parsing before filtering

for name, df in {
    "Primary Care": hpsa_pc_2022,
    "Mental Health": hpsa_mh_2022,
    "Dental Health": hpsa_dh_2022,
}.items():

    print(f"\n{name}")

    print(
        "Missing designation dates:",
        df["designation_date"].isna().sum()
    )


Primary Care
Missing designation dates: 0

Mental Health
Missing designation dates: 0

Dental Health
Missing designation dates: 0


In [113]:
# filter to qualifying 2022 designations
# create another helper:

def filter_hpsa_2022(df):
    return df[
        df["active_2022_eoy"]
        &
        df["eligible_designation_type"]
        &
        df["fips"].notna()
    ].copy()

In [114]:
# then,

pc_active_2022 = filter_hpsa_2022(
    hpsa_pc_2022
)

mh_active_2022 = filter_hpsa_2022(
    hpsa_mh_2022
)

dh_active_2022 = filter_hpsa_2022(
    hpsa_dh_2022
)

### Important: deduplicate HPSA-county combinations
#### One HPSA can have multiple component rows within the same county.

In [115]:
def deduplicate_hpsa_county(df):
    return (
        df
        .drop_duplicates(
            subset=[
                "HPSA ID",
                "fips"
            ]
        )
        .copy()
    )

In [116]:
# apply:

pc_active_2022 = deduplicate_hpsa_county(
    pc_active_2022
)

mh_active_2022 = deduplicate_hpsa_county(
    mh_active_2022
)

dh_active_2022 = deduplicate_hpsa_county(
    dh_active_2022
)

#### This is critical.
##### Otherwise the same designation may get counted multiple times within the same county.

In [117]:
# compare reconstructed coverage to PLACES
# use our existing: (keep commented unless you want a long list
# places_fips

In [118]:
# then

def compare_hpsa_coverage(df, name):
    hpsa_fips = set(df["fips"])

    print(f"\n{name}")

    print(
        "Counties with qualifying HPSA:",
        len(hpsa_fips & places_fips)
    )

    print(
        "HPSA geographies outside PLACES:",
        len(hpsa_fips - places_fips)
    )

    print(
        "PLACES counties without qualifying HPSA:",
        len(places_fips - hpsa_fips)
    )

In [119]:
# let's run it: 

compare_hpsa_coverage(
    pc_active_2022,
    "Primary Care"
)

compare_hpsa_coverage(
    mh_active_2022,
    "Mental Health"
)

compare_hpsa_coverage(
    dh_active_2022,
    "Dental Health"
)


Primary Care
Counties with qualifying HPSA: 2696
HPSA geographies outside PLACES: 114
PLACES counties without qualifying HPSA: 448

Mental Health
Counties with qualifying HPSA: 2799
HPSA geographies outside PLACES: 138
PLACES counties without qualifying HPSA: 345

Dental Health
Counties with qualifying HPSA: 2252
HPSA geographies outside PLACES: 73
PLACES counties without qualifying HPSA: 892


### Important caution about HPSA Score:

"HPSA Score
HPSA Designation Last Update Date"

Many of the designations that were active in 2022 were updated after 2022.

That means:
the designation period can be reconstructed historically,

but:
the HPSA Score in the 2026 extract may not necessarily equal the score the HPSA had in 2022.

So do not casually label it:

`hpsa_score_2022`

yet.

You can still retain the score as a candidate variable, but describe it as:

score recorded in the current extract for a designation that appears to have been active in 2022.

That distinction matters.

### 6.5 Aggregate to County Level

Qualifying HPSA designations are aggregated from designation-level records to county-level features.

For each discipline, the primary audit-stage features are:

- whether the county had at least one qualifying designation,
- the number of unique qualifying HPSA designations affecting the county, and
- the maximum HPSA score recorded among qualifying designations.

Counties with no qualifying designation are assigned zero for the binary and count variables rather than treated as missing.

**HPSA shortage/FTE and estimated underserved-population fields are not summed at this stage because designation-level values may be repeated across multiple county components and could overstate county-level shortage.**

In [120]:
# create aggregate function

def aggregate_hpsa_county(
    df,
    prefix
):
    """
    Aggregate deduplicated HPSA-county records
    to one row per county.
    """

    county = (
        df
        .groupby("fips")
        .agg(
            hpsa_count=(
                "HPSA ID",
                "nunique"
            ),
            hpsa_max_score=(
                "HPSA Score",
                "max"
            )
        )
        .reset_index()
    )

    county[
        "hpsa_any"
    ] = 1

    county = county.rename(
        columns={
            "hpsa_count":
                f"{prefix}_hpsa_count",
            "hpsa_max_score":
                f"{prefix}_hpsa_max_score",
            "hpsa_any":
                f"{prefix}_hpsa_any"
        }
    )

    return county

In [121]:
# create each table:

pc_hpsa_county = aggregate_hpsa_county(
    pc_active_2022,
    "pc"
)

mh_hpsa_county = aggregate_hpsa_county(
    mh_active_2022,
    "mh"
)

dh_hpsa_county = aggregate_hpsa_county(
    dh_active_2022,
    "dh"
)

In [122]:
# inspect

pc_hpsa_county.head()

,fips,pc_hpsa_count,pc_hpsa_max_score,pc_hpsa_any
0,01001,3,16,1
1,01003,1,15,1
2,01005,1,19,1
3,01007,1,10,1
4,01009,1,11,1


In [123]:
# restrict to PLACES geography
# we want one row for all 3,144 counties, including counties with no designation
# let's start with a county universe

county_universe = (
    places_target[
        ["fips"]
    ]
    .drop_duplicates()
    .copy()
)

In [124]:
# merge

hpsa_county = (
    county_universe
    .merge(
        pc_hpsa_county,
        on = "fips",
        how = "left",
        validate = "one_to_one"
    )
    .merge(
        mh_hpsa_county,
        on = "fips",
        how = "left",
        validate= "one_to_one"
    )
    .merge(
        dh_hpsa_county,
        on = "fips",
        how = "left",
        validate = "one_to_one"
    )
)

In [125]:
# fill true “no designation” values

# for the binary/count variables:

zero_fill_cols = [
    "pc_hpsa_any",
    "pc_hpsa_count",
    "mh_hpsa_any",
    "mh_hpsa_count",
    "dh_hpsa_any",
    "dh_hpsa_count",
]

hpsa_county[
    zero_fill_cols
] = (
    hpsa_county[
        zero_fill_cols
    ]
    .fillna(0)
)

In [126]:
# convert indicators/counts:

hpsa_county[
    zero_fill_cols
] = hpsa_county[
    zero_fill_cols
].astype(int)

#### Do not automatically fill HPSA score with zero yet

A missing max score can mean: no qualifying HPSA designation existed.

During the audit we will be leaving as NaN because

`score = 0`

and

`no HPSA designation`

are not necessarily conceptually identical.

You already have `*_hpsa_any` to distinguish designation presence.
So leave score missing for now.

In [127]:
# validate county-level HPSA dataset

print(
    "HPSA county rows:",
    len(hpsa_county)
)

print(
    "Unique county FIPS:",
    hpsa_county["fips"].nunique()
)

print(
    "Duplicate county FIPS:",
    hpsa_county["fips"]
    .duplicated()
    .sum()
)

HPSA county rows: 3144
Unique county FIPS: 3144
Duplicate county FIPS: 0


In [128]:
# check prevalence of shortage designations

summary = pd.DataFrame({
    "discipline": [
        "Primary Care",
        "Mental Health",
        "Dental Health"
    ],
    "counties_with_hpsa": [
        hpsa_county[
            "pc_hpsa_any"
        ].sum(),
        hpsa_county[
            "mh_hpsa_any"
        ].sum(),
        hpsa_county[
            "dh_hpsa_any"
        ].sum()
    ]
})

summary["pct_counties"] = (
    summary[
        "counties_with_hpsa"
    ]
    / len(hpsa_county)
    * 100
).round(1)

summary

,discipline,counties_with_hpsa,pct_counties
0,Primary Care,2696,85.8
1,Mental Health,2799,89.0
2,Dental Health,2252,71.6


In [129]:
# a useful final validation

hpsa_county[
    [
        "pc_hpsa_any",
        "mh_hpsa_any",
        "dh_hpsa_any"
    ]
].describe()

,pc_hpsa_any,mh_hpsa_any,dh_hpsa_any
count,3144.000000,3144.000000,3144.000000
mean,0.857506,0.890267,0.716285
std,0.349611,0.312606,0.450872
min,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000


In [130]:
# and

hpsa_county.head(10)

,fips,pc_hpsa_count,pc_hpsa_max_score,pc_hpsa_any,mh_hpsa_count,mh_hpsa_max_score,mh_hpsa_any,dh_hpsa_count,dh_hpsa_max_score,dh_hpsa_any
0,06045,8,17.0,1,1,19.0,1,5,18.0,1
1,02063,1,13.0,1,0,NaN,0,0,NaN,0
2,04007,5,21.0,1,6,20.0,1,6,24.0,1
3,05087,1,12.0,1,2,15.0,1,1,18.0,1
4,05067,0,NaN,0,1,18.0,1,1,11.0,1
5,05113,1,16.0,1,1,18.0,1,1,18.0,1
6,05083,1,15.0,1,1,18.0,1,0,NaN,0
7,01023,1,15.0,1,1,18.0,1,1,19.0,1
8,04013,26,20.0,1,27,20.0,1,27,25.0,1
9,02090,1,15.0,1,1,16.0,1,1,16.0,1


### HPSA Audit Result

The HPSA source files contain designation-level rather than county-level records. Multiple rows can represent components of the same HPSA, and a single designation may span multiple counties.

For the primary county-level representation, the audit retained Geographic HPSA, High Needs Geographic HPSA, and HPSA Population designations and excluded facility-specific designation types.

Using designation and withdrawal dates, an approximate year-end 2022 snapshot was reconstructed. A designation was treated as active if it had been designated by December 31, 2022 and had either not been withdrawn or was withdrawn after that date.

Repeated HPSA-county combinations were deduplicated using `HPSA ID + county FIPS` before aggregation.

After restricting to the 3,144-county PLACES geographic universe:

- approximately 2,696 counties had at least one qualifying Primary Care HPSA,
- approximately 2,799 had at least one qualifying Mental Health HPSA, and
- approximately 2,252 had at least one qualifying Dental Health HPSA.

Counties without qualifying HPSA designations are represented as zero for binary/count measures rather than treated as missing.

HPSA scores are retained as candidate features but require caution because the current extract may contain scores updated after 2022 even when the underlying designation was active during 2022.

**Decision:** HPSA data can be reduced to a valid one-row-per-county representation for integration with PLACES, AHRF, and SVI. Final feature selection and treatment of HPSA score will be determined during the cleaning and feature-engineering stage.

---

## 7. Geographic Alignment

The project uses U.S. counties and county equivalents as the common unit of analysis.

The CDC PLACES routine-checkup dataset defines the project's reference geographic universe because it contains the supervised-learning outcome. After removal of the national U.S. aggregate, the PLACES target contains 3,144 unique county/county-equivalent FIPS codes.

This section compares the geographic coverage of PLACES, SVI, AHRF, and the county-level HPSA representation to ensure that all sources can be integrated consistently.

County FIPS codes, rather than county names, are used as the authoritative geographic join key because naming conventions can differ across data sources.

### 7.1 County Coverage by Source

Each source is compared with the 3,144-county PLACES geographic universe. For each dataset, the audit records:

- number of source rows,
- number of unique FIPS codes,
- number of project counties represented,
- number of project counties missing from the source, and
- number of source geographies outside the project universe.

In [131]:
# establish the project universe
# we have `places_fips`, but we are going to be the definition explicit here:

# PLACES defines the project's county/county-equivalent universe
project_fips = set(
    places_target["fips"].dropna().unique()
)

print("Project geographic universe:", len(project_fips))

Project geographic universe: 3144


In [132]:
# create a reusable coverage function

def geographic_coverage_summary(
    df,
    source_name,
    project_fips,
    fips_col = "fips"
):
    """
    Summarize geographic coverage relative to the
    PLACES-defined project county universe.
    """

    source_fips = set(
        df[fips_col]
        .dropna()
        .astype(str)
        .unique()
    )

    return {
        "source": source_name,
        "source_rows": len(df),
        "unique_fips": len(source_fips),
        "project_fips_matched":
            len(source_fips & project_fips),
        "project_fips_missing":
            len(project_fips - source_fips),
        "extra_source_fips":
            len(source_fips - project_fips),
    }

In [133]:
# compare the sources
# for HPSA, use your county-level aggregate, not the raw designation files

coverage_results = [
    geographic_coverage_summary(
        places_target,
        "CDC PLACES",
        project_fips
    ),

    geographic_coverage_summary(
        svi,
        "CDC/ATSDR SVI",
        project_fips
    ),

    geographic_coverage_summary(
        ahrf,
        "HRSA AHRF - raw geography",
        project_fips
    ),

    geographic_coverage_summary(
        ahrf_county,
        "HRSA AHRF - project aligned",
        project_fips
    ),

    geographic_coverage_summary(
        hpsa_county,
        "HRSA HPSA - county aggregate",
        project_fips
    ),
]

geographic_coverage = pd.DataFrame(
    coverage_results
)

geographic_coverage

,source,source_rows,unique_fips,project_fips_matched,project_fips_missing,extra_source_fips
0,CDC PLACES,3144,3144,3144,0,0
1,CDC/ATSDR SVI,3144,3144,3144,0,0
2,HRSA AHRF - raw geography,3240,3240,3144,0,96
3,HRSA AHRF - project aligned,3144,3144,3144,0,0
4,HRSA HPSA - county aggregate,3144,3144,3144,0,0


#### Important HPSA interpretation

The HPSA table has 3,144 rows **because we intentionally created a county-level table based on the PLACES universe**.

That does not mean every county had an HPSA designation.

In [134]:
# check 

hpsa_designation_coverage = pd.DataFrame({
    "discipline": [
        "Primary Care",
        "Mental Health",
        "Dental Health"
    ],

    "counties_with_designation": [
        hpsa_county["pc_hpsa_any"].sum(),
        hpsa_county["mh_hpsa_any"].sum(),
        hpsa_county["dh_hpsa_any"].sum(),
    ]
})

hpsa_designation_coverage[
    "pct_project_counties"
] = (
    hpsa_designation_coverage[
        "counties_with_designation"
    ]
    / len(project_fips)
    * 100
).round(1)

hpsa_designation_coverage

,discipline,counties_with_designation,pct_project_counties
0,Primary Care,2696,85.8
1,Mental Health,2799,89.0
2,Dental Health,2252,71.6


This distinguishes:
> **geographic coverage**

from:
> **having a shortage designation**

A county with:

`pc_hpsa_any = 0` is still a valid county observation.

It just did not have a qualifying reconstructed Primary Care HPSA.

### 7.2 Unmatched FIPS Codes

FIPS differences are examined explicitly rather than allowing unmatched geographic records to be silently removed during merging.

PLACES serves as the reference geography. Extra records in another source are documented before the source is restricted to the project universe.

In [135]:
# confirm PLACES ↔ SVI again

svi_fips = set(
    svi["fips"].dropna().unique()
)

print(
    "PLACES not in SVI:",
    len(project_fips - svi_fips)
)

print(
    "SVI not in PLACES:",
    len(svi_fips - project_fips)
)

PLACES not in SVI: 0
SVI not in PLACES: 0


In [136]:
# AHRF unmatched geography
# we already know that AHRF contains extras, but we will summarize them here: 

ahrf_fips = set(
    ahrf["fips"]
    .dropna()
    .unique()
)

ahrf_extra_fips = sorted(
    ahrf_fips - project_fips
)

ahrf_missing_fips = sorted(
    project_fips - ahrf_fips
)

print(
    "Project counties missing from AHRF:",
    len(ahrf_missing_fips)
)

print(
    "AHRF geographies outside project universe:",
    len(ahrf_extra_fips)
)

Project counties missing from AHRF: 0
AHRF geographies outside project universe: 96


In [137]:
# inspect the AHRF extras

ahrf_extra_records = (
    ahrf[
        ahrf["fips"].isin(
            ahrf_extra_fips
        )
    ][
        [
            "fips",
            "st_name",
            "cnty_name"
        ]
    ]
    .sort_values(
        [
            "st_name",
            "cnty_name"
        ]
    )
)

ahrf_extra_records.head(20)

,fips,st_name,cnty_name
91,02201,Alaska,Pr of Wales-OutrKetch(CA)
94,02232,Alaska,Skagway-Hoonah-Angoon(CA)
96,02261,Alaska,Valdez-Cordova (CA)
98,02280,Alaska,Wrangell-Petersburg (CA)
313,09001,Connecticut,Fairfield
314,09003,Connecticut,Hartford
315,09005,Connecticut,Litchfield
316,09007,Connecticut,Middlesex
317,09009,Connecticut,New Haven
318,09011,Connecticut,New London


Our earlier audit indicated these include records associated with:
- Puerto Rico
- U.S. Virgin Islands
- Guam
- historical Connecticut counties
- historical Alaska county equivalents
- former Virginia independent-city geography
This is why the full AHRF source contains more rows than the project universe.

In [138]:
# inspect HPSA source geographies outside PLACES
# our raw reconstructed HPSA tables may also contain territories

hpsa_source_tables = {
    "Primary Care": pc_active_2022,
    "Mental Health": mh_active_2022,
    "Dental Health": dh_active_2022,
}

for name, df in hpsa_source_tables.items():

    source_fips = set(
        df["fips"]
        .dropna()
        .unique()
    )

    print(f"\n{name}")

    print(
        "Qualifying FIPS in project universe:",
        len(source_fips & project_fips)
    )

    print(
        "Qualifying FIPS outside project universe:",
        len(source_fips - project_fips)
    )


Primary Care
Qualifying FIPS in project universe: 2696
Qualifying FIPS outside project universe: 114

Mental Health
Qualifying FIPS in project universe: 2799
Qualifying FIPS outside project universe: 138

Dental Health
Qualifying FIPS in project universe: 2252
Qualifying FIPS outside project universe: 73


> HPSA was deliberately aggregated onto the PLACES universe after qualifying designations were identified.

### 7.3 Known Geography Issues

Several geographic differences were identified during the source audits.

#### PLACES national aggregate

The initial PLACES routine-checkup subset contained one national U.S. aggregate in addition to county observations. The record had `StateAbbr = "US"` and did not represent a valid county FIPS. It was excluded because the project's unit of analysis is the county/county equivalent.

#### AHRF additional geographies

The complete AHRF source contains 3,240 unique geographic records, compared with the 3,144-county PLACES universe. All 3,144 project counties are represented in AHRF; the additional records include territories and historical county/county-equivalent geographies.

These records are documented but excluded from the project-aligned AHRF dataset.

#### Connecticut geography transition

PLACES and SVI use the current Connecticut planning-region county equivalents. AHRF contains matching records for these geographies, so the FIPS join succeeds; however, several 2022 healthcare-resource variables are missing because some underlying AHRF sources still reflected Connecticut's former county geography during that period.

This is therefore a feature-availability issue rather than a failed geographic match.

#### HPSA geography

The raw HPSA files contain designation-level geographic components rather than one county record per row. Some raw records also represent geographic units outside the PLACES project universe or contain non-county identifiers.

Qualifying HPSA records were therefore standardized, deduplicated by HPSA ID and county FIPS, and aggregated onto the 3,144-county PLACES geographic universe.

#### County naming differences

County names differ slightly across sources—for example, one dataset may use `Mendocino`, while another uses `Mendocino County`. County names are therefore retained for interpretation only and are not used as merge keys.

Five-character county FIPS codes are used as the authoritative geographic identifier.

In [139]:
# create a master geographic alignment table
# this is useful because it gives us one table that proves each project county 
# is represented in the aligned sources

# Start with PLACES:

geographic_alignment = (
    places_target[
        [
            "fips",
            "StateAbbr",
            "LocationName"
        ]
    ]
    .drop_duplicates()
    .copy()
)

In [140]:
# add SVI presence:

geographic_alignment = (
    geographic_alignment
    .merge(
        svi[
            ["fips"]
        ]
        .drop_duplicates()
        .assign(
            in_svi=True
        ),
        on="fips",
        how="left",
        validate="one_to_one"
    )
)

In [141]:
# add AHRF:

geographic_alignment = (
    geographic_alignment
    .merge(
        ahrf_county[
            ["fips"]
        ]
        .drop_duplicates()
        .assign(
            in_ahrf=True
        ),
        on="fips",
        how="left",
        validate="one_to_one"
    )
)

In [142]:
# add HPSA:

geographic_alignment = (
    geographic_alignment
    .merge(
        hpsa_county[
            ["fips"]
        ]
        .drop_duplicates()
        .assign(
            in_hpsa_table=True
        ),
        on="fips",
        how="left",
        validate="one_to_one"
    )
)

In [143]:
# fill flags:

presence_cols = [
    "in_svi",
    "in_ahrf",
    "in_hpsa_table"
]

geographic_alignment[
    presence_cols
] = (
    geographic_alignment[
        presence_cols
    ]
    .fillna(False)
)

In [144]:
# then, 

geographic_alignment[
    presence_cols
].value_counts()

in_svi  in_ahrf  in_hpsa_table
True    True     True             3144
Name: count, dtype: int64

In [145]:
# one final validation assertion
# useful if geography changes later

assert len(
    geographic_alignment
) == 3144

assert (
    geographic_alignment[
        "fips"
    ].nunique()
    == 3144
)

assert geographic_alignment[
    presence_cols
].all().all()

print(
    "Geographic alignment validation passed."
)

Geographic alignment validation passed.


### Geographic Alignment Result

The project geographic universe contains 3,144 U.S. counties and county equivalents, defined by the county-level CDC PLACES routine-checkup target.

After FIPS standardization:

- PLACES contains 3,144 project counties;
- SVI contains the same 3,144 counties with complete overlap;
- AHRF contains all 3,144 project counties plus 96 additional geographic/historical records, which were documented and excluded from the project-aligned subset; and
- the aggregated HPSA table contains one row for each of the 3,144 project counties, with zero-valued designation indicators/counts representing counties without qualifying HPSA designations.

No project county is currently lost because of a failed geographic join.

Remaining geographic concerns primarily involve feature availability rather than geographic coverage, particularly the transition from historical Connecticut counties to current planning-region county equivalents.

**Decision:** The 3,144 PLACES county/county-equivalent FIPS codes will serve as the authoritative geographic universe for all subsequent project integration.

---

## 8. Incremental Merge Audit

The four project data sources are merged incrementally using standardized five-character county FIPS codes.

CDC PLACES serves as the base dataset because it defines both the supervised-learning target and the project's 3,144-county geographic universe.

Each merge is performed as a left one-to-one join and validated before the next source is added. This allows unexpected row loss, row multiplication, or geographic mismatches to be identified immediately rather than after all sources have been combined.

The merge sequence is:

1. PLACES + AHRF
2. Add HPSA
3. Add SVI

No imputation or final feature engineering is performed during this stage.

### create the PLACES base table
You don't need all of the original PLACES columns in the eventual master table.

In [146]:
# create a clean base with identifiers and your target:

places_base = (
    places_target[
        [
            "fips",
            "StateAbbr",
            "LocationName",
            "Year",
            "routine_checkup_pct"
        ]
    ]
    .copy()
)

In [147]:
# validate

print("PLACES base shape:", places_base.shape)
print("Unique FIPS:", places_base["fips"].nunique())
print("Duplicate FIPS:", places_base["fips"].duplicated().sum())

PLACES base shape: (3144, 5)
Unique FIPS: 3144
Duplicate FIPS: 0


In [148]:
# rename the identifier fields so their source is obvious:

places_base = places_base.rename(
    columns={
        "StateAbbr": "state_abbr",
        "LocationName": "county_name_places",
        "Year": "places_target_year"
    }
)

In [149]:
# check 

places_base.head()

,fips,state_abbr,county_name_places,places_target_year,routine_checkup_pct
615,06045,CA,Mendocino,2022,68.6
736,02063,AK,Chugach,2022,62.2
789,04007,AZ,Gila,2022,68.8
794,05087,AR,Madison,2022,74.5
867,05067,AR,Jackson,2022,76.7


### 8.1 Merge PLACES and AHRF

AHRF is added first because it provides the primary healthcare-resource predictors for the supervised analysis.

Only the project-aligned AHRF dataset is used. This subset contains one row for each of the 3,144 PLACES counties/county equivalents.

In [150]:
# select only the AHRF columns we actually audited
# we already have `ahrf_feature_cols`
# which should contain our candidate AHRF fields except the original FIPS variable

In [151]:
# create the merge table:

ahrf_merge = (
    ahrf_county[
        ["fips"] + ahrf_feature_cols
    ]
    .copy()
)

In [152]:
# check 

print("AHRF merge rows:", len(ahrf_merge))
print("Unique FIPS:", ahrf_merge["fips"].nunique())
print("Duplicate FIPS:", ahrf_merge["fips"].duplicated().sum())

AHRF merge rows: 3144
Unique FIPS: 3144
Duplicate FIPS: 0


In [153]:
# perform the merge

master_audit = places_base.merge(
    ahrf_merge,
    on="fips",
    how="left",
    validate="one_to_one",
    indicator="ahrf_merge_status"
)

In [154]:
# then,

print("Rows before AHRF merge:", len(places_base))
print("Rows after AHRF merge:", len(master_audit))

print("\nAHRF merge status:")
print(master_audit["ahrf_merge_status"].value_counts())

Rows before AHRF merge: 3144
Rows after AHRF merge: 3144

AHRF merge status:
ahrf_merge_status
both          3144
left_only        0
right_only       0
Name: count, dtype: int64


Because this is a left join, `right_only` generally won't appear, but the indicator still gives useful confirmation.

In [155]:
# assert that nothing changed structurally

assert len(master_audit) == 3144
assert master_audit["fips"].nunique() == 3144
assert master_audit["fips"].duplicated().sum() == 0
assert (master_audit["ahrf_merge_status"] == "both").all()

print("PLACES + AHRF merge validation passed.")

PLACES + AHRF merge validation passed.


In [156]:
# then remove the temporary indicator:

master_audit = master_audit.drop(
    columns="ahrf_merge_status"
)

### 8.2 Add County-Level HPSA Features

The county-level HPSA table is added after AHRF.

The HPSA source was previously reconstructed and aggregated to the same 3,144-county PLACES universe. Counties without qualifying shortage designations remain valid observations and are represented with zero-valued designation indicators/counts.

In [157]:
# inspect HPSA columns first

hpsa_county.columns.tolist()

['fips',
 'pc_hpsa_count',
 'pc_hpsa_max_score',
 'pc_hpsa_any',
 'mh_hpsa_count',
 'mh_hpsa_max_score',
 'mh_hpsa_any',
 'dh_hpsa_count',
 'dh_hpsa_max_score',
 'dh_hpsa_any']

In [158]:
# merge 

master_audit = master_audit.merge(
    hpsa_county,
    on = "fips",
    how = "left",
    validate = "one_to_one",
    indicator = "hpsa_merge_status"
)

In [159]:
# check

print("Rows after HPSA merge:", len(master_audit))

print("\nHPSA merge status:")
print(master_audit["hpsa_merge_status"].value_counts())

Rows after HPSA merge: 3144

HPSA merge status:
hpsa_merge_status
both          3144
left_only        0
right_only       0
Name: count, dtype: int64


In [160]:
# validate

assert len(master_audit) == 3144
assert master_audit["fips"].nunique() == 3144
assert master_audit["fips"].duplicated().sum() == 0
assert (master_audit["hpsa_merge_status"] == "both").all()

print("HPSA merge validation passed.")

HPSA merge validation passed.


In [161]:
# drop temporary indicator

master_audit = master_audit.drop(
    columns="hpsa_merge_status"
)

In [162]:
# sanity-check HPSA indicators

master_audit[
    [
        "pc_hpsa_any",
        "mh_hpsa_any",
        "dh_hpsa_any"
    ]
].sum()

pc_hpsa_any    2696
mh_hpsa_any    2799
dh_hpsa_any    2252
dtype: int64

### Note: 
A county may exist in the dataset while correctly having: 

`hpsa_any = 0`

### 8.3 Add SVI Variables

The 2022 SVI is added last.

Only the underlying social-vulnerability indicators currently relevant to the project are added at this stage. Composite SVI rankings are not included simultaneously with their component variables because doing so would introduce redundant representations of the same underlying information.

Final clustering-feature selection will occur after EDA.

In [163]:
# define candidate SVI fields

svi_feature_cols = [
    # Theme 1: Socioeconomic Status
    "EP_POV150",
    "EP_UNEMP",
    "EP_HBURD",
    "EP_NOHSDP",
    "EP_UNINSUR",

    # Theme 2: Household Characteristics
    "EP_AGE65",
    "EP_AGE17",
    "EP_DISABL",
    "EP_SNGPNT",
    "EP_LIMENG",

    # Theme 3: Racial & Ethnic Minority Status
    "EP_MINRTY",

    # Theme 4: Housing Type & Transportation
    "EP_MUNIT",
    "EP_MOBILE",
    "EP_CROWD",
    "EP_NOVEH",
    "EP_GROUPQ"
]

In [164]:
# verify that all exist:

svi_validation = pd.DataFrame({
    "variable": svi_feature_cols,
    "found_in_svi": [
        col in svi.columns
        for col in svi_feature_cols
    ]
})

svi_validation

,variable,found_in_svi
0,EP_POV150,True
1,EP_UNEMP,True
2,EP_HBURD,True
3,EP_NOHSDP,True
4,EP_UNINSUR,True
5,EP_AGE65,True
6,EP_AGE17,True
7,EP_DISABL,True
8,EP_SNGPNT,True
9,EP_LIMENG,True


In [165]:
# then,

assert svi_validation["found_in_svi"].all()

print("All candidate SVI variables found.")

All candidate SVI variables found.


In [166]:
# create SVI merge table

svi_merge = (
    svi[
        ["fips"] + svi_feature_cols
    ]
    .copy()
)

In [167]:
# check

print("SVI merge rows:", len(svi_merge))
print("Unique FIPS:", svi_merge["fips"].nunique())
print("Duplicate FIPS:", svi_merge["fips"].duplicated().sum())

SVI merge rows: 3144
Unique FIPS: 3144
Duplicate FIPS: 0


In [168]:
# merge

master_audit = master_audit.merge(
    svi_merge,
    on = "fips",
    how = "left",
    validate = "one_to_one",
    indicator = "svi_merge_status"
)

In [169]:
# then, 

print("Rows after SVI merge:", len(master_audit))

print("\nSVI merge status:")
print(master_audit["svi_merge_status"].value_counts())

Rows after SVI merge: 3144

SVI merge status:
svi_merge_status
both          3144
left_only        0
right_only       0
Name: count, dtype: int64


In [170]:
# validate

assert len(master_audit) == 3144
assert master_audit["fips"].nunique() == 3144
assert master_audit["fips"].duplicated().sum() == 0
assert (master_audit["svi_merge_status"] == "both").all()

print("SVI merge validation passed.")

SVI merge validation passed.


In [171]:
# then, 

master_audit = master_audit.drop(
    columns="svi_merge_status"
)

### 8.4 Final Structural Validation

After all four sources are combined, the audit verifies that the integrated dataset still contains exactly one observation per project county and that no rows were lost or duplicated during merging.

In [172]:
# check final dimensions

print("Master audit dataset shape:", master_audit.shape)

print(
    "Unique county FIPS:",
    master_audit["fips"].nunique()
)

print(
    "Duplicate county FIPS:",
    master_audit["fips"].duplicated().sum()
)

Master audit dataset shape: (3144, 41)
Unique county FIPS: 3144
Duplicate county FIPS: 0


In [173]:
# final assertions

assert len(master_audit) == 3144
assert master_audit["fips"].nunique() == 3144
assert master_audit["fips"].duplicated().sum() == 0

print(
    "Final structural validation passed: "
    "one row per project county."
)

Final structural validation passed: one row per project county.


In [174]:
# build a merge-audit summary table
# provides a concise record of the whole integration process

merge_audit_summary = pd.DataFrame({
    "stage": [
        "PLACES base",
        "PLACES + AHRF",
        "PLACES + AHRF + HPSA",
        "PLACES + AHRF + HPSA + SVI"
    ],
    "rows": [
        len(places_base),
        3144,
        3144,
        len(master_audit)
    ],
    "unique_fips": [
        places_base["fips"].nunique(),
        3144,
        3144,
        master_audit["fips"].nunique()
    ],
    "rows_lost": [
        0,
        0,
        0,
        0
    ]
})

merge_audit_summary

,stage,rows,unique_fips,rows_lost
0,PLACES base,3144,3144,0
1,PLACES + AHRF,3144,3144,0
2,PLACES + AHRF + HPSA,3144,3144,0
3,PLACES + AHRF + HPSA + SVI,3144,3144,0


### We could later improve this by calculating values dynamically, but this is fine for the audit notebook if each earlier merge was explicitly validated.

In [175]:
# inspect the integrated table

master_audit.head()

,fips,state_abbr,county_name_places,places_target_year,routine_checkup_pct,popn_est_22,phys_nf_prim_care_pc_exc_rsdt_22,np_npi_22,pa_npi_22,hosp_22,...,EP_AGE17,EP_DISABL,EP_SNGPNT,EP_LIMENG,EP_MINRTY,EP_MUNIT,EP_MOBILE,EP_CROWD,EP_NOVEH,EP_GROUPQ
0,06045,CA,Mendocino,2022,68.6,89783.0,72.0,75.0,29.0,3.0,...,21.1,18.7,7.0,4.4,37.2,4.4,9.9,4.9,8.0,2.7
1,02063,AK,Chugach,2022,62.2,6874.0,10.0,1.0,2.0,0.0,...,21.7,15.9,5.4,1.7,29.5,10.1,15.6,5.6,5.7,3.6
2,04007,AZ,Gila,2022,68.8,53922.0,27.0,29.0,21.0,2.0,...,19.4,21.2,3.6,0.6,38.8,1.2,19.8,4.4,6.9,2.3
3,05087,AR,Madison,2022,74.5,17486.0,4.0,9.0,0.0,0.0,...,23.4,19.8,6.3,0.0,24.2,1.1,18.2,3.5,3.2,0.5
4,05067,AR,Jackson,2022,76.7,16624.0,5.0,17.0,2.0,1.0,...,19.5,23.2,7.7,0.1,24.1,0.9,9.6,0.4,8.0,13.7


In [176]:
# and

master_audit.columns.tolist()

['fips',
 'state_abbr',
 'county_name_places',
 'places_target_year',
 'routine_checkup_pct',
 'popn_est_22',
 'phys_nf_prim_care_pc_exc_rsdt_22',
 'np_npi_22',
 'pa_npi_22',
 'hosp_22',
 'hosp_beds_22',
 'fedly_qualfd_hlth_ctr_22',
 'rural_hlth_clincs_22',
 'rural_urban_contnm_23',
 'popn_densty_per_squr_mi_20',
 'pers_noins_lt65_pct_21',
 'pc_hpsa_count',
 'pc_hpsa_max_score',
 'pc_hpsa_any',
 'mh_hpsa_count',
 'mh_hpsa_max_score',
 'mh_hpsa_any',
 'dh_hpsa_count',
 'dh_hpsa_max_score',
 'dh_hpsa_any',
 'EP_POV150',
 'EP_UNEMP',
 'EP_HBURD',
 'EP_NOHSDP',
 'EP_UNINSUR',
 'EP_AGE65',
 'EP_AGE17',
 'EP_DISABL',
 'EP_SNGPNT',
 'EP_LIMENG',
 'EP_MINRTY',
 'EP_MUNIT',
 'EP_MOBILE',
 'EP_CROWD',
 'EP_NOVEH',
 'EP_GROUPQ']

In [177]:
# inspect a few fields from each source together:

master_audit[
    [
        "fips",
        "state_abbr",
        "county_name_places",
        "routine_checkup_pct",

        # AHRF
        "popn_est_22",
        "phys_nf_prim_care_pc_exc_rsdt_22",

        # HPSA
        "pc_hpsa_any",
        "pc_hpsa_count",
        "pc_hpsa_max_score",

        # SVI
        "EP_POV150",
        "EP_UNINSUR",
        "EP_NOVEH"
    ]
].head(10)

,fips,state_abbr,county_name_places,routine_checkup_pct,popn_est_22,phys_nf_prim_care_pc_exc_rsdt_22,pc_hpsa_any,pc_hpsa_count,pc_hpsa_max_score,EP_POV150,EP_UNINSUR,EP_NOVEH
0,06045,CA,Mendocino,68.6,89783.0,72.0,1,8,17.0,25.4,9.2,8.0
1,02063,AK,Chugach,62.2,6874.0,10.0,1,1,13.0,10.6,14.7,5.7
2,04007,AZ,Gila,68.8,53922.0,27.0,1,5,21.0,30.6,9.8,6.9
3,05087,AR,Madison,74.5,17486.0,4.0,1,1,12.0,30.1,10.4,3.2
4,05067,AR,Jackson,76.7,16624.0,5.0,0,0,NaN,36.0,8.4,8.0
5,05113,AR,Polk,75.0,19337.0,15.0,1,1,16.0,30.9,9.5,5.3
6,05083,AR,Logan,75.9,21253.0,4.0,1,1,15.0,29.0,6.7,5.2
7,01023,AL,Choctaw,78.4,12439.0,5.0,1,1,15.0,31.9,8.2,5.7
8,04013,AZ,Maricopa,71.3,4551524.0,3140.0,1,26,20.0,19.0,11.0,5.2
9,02090,AK,Fairbanks North Star,65.7,95356.0,85.0,1,1,15.0,13.4,8.6,4.8


### Simply confirming:
#### Values from four distinct sources now coexist correctly on the same county row.

In [178]:
# verify target completeness

print(
    "Missing routine-checkup target:",
    master_audit[
        "routine_checkup_pct"
    ].isna().sum()
)

Missing routine-checkup target: 0


In [179]:
# check source-specific feature counts
# helps verify that nothing unexpectedly disappeared

print(
    "AHRF candidate features:",
    len(ahrf_feature_cols)
)

print(
    "HPSA features:",
    len([
        col
        for col in master_audit.columns
        if "_hpsa_" in col
    ])
)

print(
    "SVI candidate features:",
    len(svi_feature_cols)
)

AHRF candidate features: 11
HPSA features: 9
SVI candidate features: 16


### Incremental Merge Audit Result

CDC PLACES was used as the base dataset because it defines both the supervised-learning target and the project's 3,144-county geographic universe.

AHRF, the reconstructed county-level HPSA features, and selected SVI variables were added sequentially using standardized five-character county FIPS codes and one-to-one left joins.

At each merge stage:

- the dataset retained 3,144 rows,
- each county FIPS remained unique,
- no PLACES counties were lost, and
- no joins produced duplicate county observations.

The resulting audit-stage table therefore contains one observation per project county with variables drawn from all four core sources.

The successful merge confirms structural compatibility across the project datasets. Remaining issues concern feature-level missingness, temporal alignment, redundancy, and feature engineering rather than geographic integration.

**Decision:** The merged audit-stage dataset will serve as the basis for the subsequent missingness audit and cleaning/feature-engineering workflow.

---

## 9. Missingness Audit

After successful integration of PLACES, AHRF, HPSA, and SVI, missingness is evaluated at the feature level and county level.

This audit distinguishes among:

- **true missingness:** a value expected for a county is unavailable,
- **structural missingness:** a blank value is expected because the underlying concept does not apply, and
- **encoded missingness:** a source uses a sentinel value rather than a standard missing-value representation.

No imputation is performed in this section. The purpose is to document missingness patterns and identify decisions that must be addressed during cleaning and feature engineering.

In [180]:
# Inspect the merged dataset as it currently exists

missing_summary = pd.DataFrame({
    "missing_n": master_audit.isna().sum(),
    "missing_pct": (
        master_audit.isna().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary[
        missing_summary["missing_n"] > 0
    ]
    .sort_values(
        "missing_n",
        ascending=False
    )
)

missing_summary

,missing_n,missing_pct
dh_hpsa_max_score,892,28.37
pc_hpsa_max_score,448,14.25
mh_hpsa_max_score,345,10.97
fedly_qualfd_hlth_ctr_22,13,0.41
rural_hlth_clincs_22,13,0.41
pers_noins_lt65_pct_21,10,0.32
np_npi_22,9,0.29
phys_nf_prim_care_pc_exc_rsdt_22,9,0.29
hosp_22,9,0.29
pa_npi_22,9,0.29


### This tells us which columns contain ordinary NaN values.

#### FYI: But do not interpret this table by itself yet, because HPSA scores and SVI sentinel values require special treatment.

### 9.1 Target Completeness

Missingness in the supervised-learning target is evaluated separately because counties without an outcome value could not be used in supervised model training or evaluation.

In [181]:
target_missing = (
    master_audit[
        "routine_checkup_pct"
    ]
    .isna()
    .sum()
)

print(
    "Missing routine-checkup target:",
    target_missing
)

Missing routine-checkup target: 0


In [182]:
# confirm the target remains numeric:

master_audit[
    "routine_checkup_pct"
].describe()

count    3144.000000
mean       73.822933
std         3.580040
min        59.300000
25%        71.600000
50%        74.400000
75%        76.400000
max        83.100000
Name: routine_checkup_pct, dtype: float64

### If result = 0:
#### All 3,144 project counties have an age-adjusted routine-checkup outcome, so target missingness does not reduce the supervised-learning sample.

### 9.2 AHRF Missingness

AHRF candidate predictors are examined separately because earlier auditing identified geographically structured missingness, particularly for Connecticut planning regions.

Missingness is evaluated by variable and by county/state to determine whether it appears random or concentrated in specific geographies.

In [183]:
# use the candidate variables we already defined:

ahrf_missing_summary = pd.DataFrame({
    "missing_n":
        master_audit[
            ahrf_feature_cols
        ].isna().sum(),

    "missing_pct":
        (
            master_audit[
                ahrf_feature_cols
            ]
            .isna()
            .mean()
            * 100
        ).round(2)
})

ahrf_missing_summary = (
    ahrf_missing_summary
    .sort_values(
        "missing_n",
        ascending=False
    )
)

ahrf_missing_summary

,missing_n,missing_pct
fedly_qualfd_hlth_ctr_22,13,0.41
rural_hlth_clincs_22,13,0.41
pers_noins_lt65_pct_21,10,0.32
hosp_beds_22,9,0.29
np_npi_22,9,0.29
phys_nf_prim_care_pc_exc_rsdt_22,9,0.29
pa_npi_22,9,0.29
popn_densty_per_squr_mi_20,9,0.29
hosp_22,9,0.29
popn_est_22,0,0.00


You should see relatively low percentages overall, with variables such as the healthcare-resource measures showing missingness concentrated in a small number of counties.

In [184]:
# identify counties with any AHRF missingness

ahrf_missing_mask = (
    master_audit[
        ahrf_feature_cols
    ]
    .isna()
    .any(axis=1)
)

ahrf_missing_counties = (
    master_audit.loc[
        ahrf_missing_mask,
        [
            "fips",
            "state_abbr",
            "county_name_places"
        ] + ahrf_feature_cols
    ]
    .copy()
)

print(
    "Counties with at least one "
    "missing AHRF candidate value:",
    len(ahrf_missing_counties)
)

Counties with at least one missing AHRF candidate value: 14


In [185]:
# summarize by state

ahrf_missing_by_state = (
    ahrf_missing_counties[
        "state_abbr"
    ]
    .value_counts()
    .rename_axis("state_abbr")
    .reset_index(
        name="counties_with_missing_ahrf"
    )
)

ahrf_missing_by_state

,state_abbr,counties_with_missing_ahrf
0,CT,9
1,AK,4
2,HI,1


In [186]:
# count missing fields per county

ahrf_missing_counties[
    "ahrf_missing_feature_count"
] = (
    ahrf_missing_counties[
        ahrf_feature_cols
    ]
    .isna()
    .sum(axis=1)
)

ahrf_missing_counties[
    [
        "fips",
        "state_abbr",
        "county_name_places",
        "ahrf_missing_feature_count"
    ]
].sort_values(
    "ahrf_missing_feature_count",
    ascending=False
)

,fips,state_abbr,county_name_places,ahrf_missing_feature_count
273,09130,CT,Lower Connecticut River Valley,9
282,09110,CT,Capitol,9
450,09120,CT,Greater Bridgeport,9
439,09140,CT,Naugatuck Valley,9
401,09180,CT,Southeastern Connecticut,9
354,09150,CT,Northeastern Connecticut,9
457,09170,CT,South Central Connecticut,9
462,09190,CT,Western Connecticut,9
465,09160,CT,Northwest Hills,9
60,02195,AK,Petersburg,2


This will make the Connecticut pattern especially obvious if several 2022 resource fields are simultaneously unavailable.

### 9.3 HPSA Structural Missingness

HPSA binary and count variables use zero to represent counties without a qualifying shortage designation.

Maximum HPSA score is treated differently. A county without a qualifying designation has no meaningful maximum score, so a missing score in that situation represents structural missingness rather than an unavailable measurement.

The audit therefore checks whether score values are missing only where no qualifying designation exists.

In [187]:
# primary care

pc_hpsa_check = pd.crosstab(
    master_audit[
        "pc_hpsa_any"
    ],
    master_audit[
        "pc_hpsa_max_score"
    ].isna(),
    rownames=[
        "pc_hpsa_any"
    ],
    colnames=[
        "score_is_missing"
    ]
)

pc_hpsa_check

score_is_missing,False,True
pc_hpsa_any,,
0,0,448
1,2696,0


In [188]:
# more directly

print(
    "PC score missing with no designation:",
    master_audit.loc[
        master_audit["pc_hpsa_any"] == 0,
        "pc_hpsa_max_score"
    ].isna().sum()
)

print(
    "PC score unexpectedly missing "
    "with designation:",
    master_audit.loc[
        master_audit["pc_hpsa_any"] == 1,
        "pc_hpsa_max_score"
    ].isna().sum()
)

PC score missing with no designation: 448
PC score unexpectedly missing with designation: 0


In [189]:
# repeat for mental and dental 

for prefix, label in [
    ("pc", "Primary Care"),
    ("mh", "Mental Health"),
    ("dh", "Dental Health")
]:

    no_designation_missing = (
        master_audit.loc[
            master_audit[
                f"{prefix}_hpsa_any"
            ] == 0,
            f"{prefix}_hpsa_max_score"
        ]
        .isna()
        .sum()
    )

    designation_missing = (
        master_audit.loc[
            master_audit[
                f"{prefix}_hpsa_any"
            ] == 1,
            f"{prefix}_hpsa_max_score"
        ]
        .isna()
        .sum()
    )

    print(f"\n{label}")
    print(
        "Structural score missingness:",
        no_designation_missing
    )
    print(
        "Unexpected score missingness:",
        designation_missing
    )


Primary Care
Structural score missingness: 448
Unexpected score missingness: 0

Mental Health
Structural score missingness: 345
Unexpected score missingness: 0

Dental Health
Structural score missingness: 892
Unexpected score missingness: 0


### How to interpret this: 

Suppose a county has: 

- `pc_hpsa_any = 0`
- `pc_hpsa_count = 0`
- `pc_hpsa_max_score = NaN`

That is not a data-quality problem.

It means:
> There is no qualifying Primary Care HPSA from which a maximum score could be calculated.

Therefore we should **not automatically impute that score as if HRSA forgot to report it**.

### 9.4 SVI Encoded Missingness

SVI source files may use sentinel values such as `-999` to represent unavailable or suppressed information.

Because these values are numeric, standard missing-value checks do not detect them. Candidate SVI variables are therefore screened explicitly for sentinel values before later cleaning or modeling.

In [190]:
# check -999 counts

svi_sentinel_summary = pd.DataFrame({
    "sentinel_-999_n": (
        master_audit[
            svi_feature_cols
        ]
        .eq(-999)
        .sum()
    )
})

svi_sentinel_summary[
    "sentinel_-999_pct"
] = (
    svi_sentinel_summary[
        "sentinel_-999_n"
    ]
    / len(master_audit)
    * 100
).round(2)

svi_sentinel_summary = (
    svi_sentinel_summary
    .sort_values(
        "sentinel_-999_n",
        ascending=False
    )
)

svi_sentinel_summary

,sentinel_-999_n,sentinel_-999_pct
EP_POV150,0,0.0
EP_UNEMP,0,0.0
EP_HBURD,0,0.0
EP_NOHSDP,0,0.0
EP_UNINSUR,0,0.0
EP_AGE65,0,0.0
EP_AGE17,0,0.0
EP_DISABL,0,0.0
EP_SNGPNT,0,0.0
EP_LIMENG,0,0.0


In [191]:
# check ordinary SVI NaNs

svi_nan_summary = pd.DataFrame({
    "missing_n":
        master_audit[
            svi_feature_cols
        ].isna().sum(),

    "missing_pct":
        (
            master_audit[
                svi_feature_cols
            ]
            .isna()
            .mean()
            * 100
        ).round(2)
})

svi_nan_summary

,missing_n,missing_pct
EP_POV150,0,0.0
EP_UNEMP,0,0.0
EP_HBURD,0,0.0
EP_NOHSDP,0,0.0
EP_UNINSUR,0,0.0
EP_AGE65,0,0.0
EP_AGE17,0,0.0
EP_DISABL,0,0.0
EP_SNGPNT,0,0.0
EP_LIMENG,0,0.0


In [192]:
# create an audit-only SVI missingness view
# if -999 values exist, it can be helpful to calculate the effective missingness 
# without changing master_audit.

svi_missingness_view = (
    master_audit[
        svi_feature_cols
    ]
    .copy()
)

svi_missingness_view = (
    svi_missingness_view
    .replace(
        -999,
        np.nan
    )
)

In [193]:
# then, 

svi_effective_missing = pd.DataFrame({
    "effective_missing_n":
        svi_missingness_view
        .isna()
        .sum(),

    "effective_missing_pct":
        (
            svi_missingness_view
            .isna()
            .mean()
            * 100
        ).round(2)
})

svi_effective_missing

,effective_missing_n,effective_missing_pct
EP_POV150,0,0.0
EP_UNEMP,0,0.0
EP_HBURD,0,0.0
EP_NOHSDP,0,0.0
EP_UNINSUR,0,0.0
EP_AGE65,0,0.0
EP_AGE17,0,0.0
EP_DISABL,0,0.0
EP_SNGPNT,0,0.0
EP_LIMENG,0,0.0


### 9.5 County-Level Missingness

Ask:

> Which counties have any true predictor missingness?

In [194]:
# start with AHRF and SVI candidate features

predictor_missingness_view = pd.concat(
    [
        master_audit[
            ahrf_feature_cols
        ],
        svi_missingness_view
    ],
    axis=1
)

In [195]:
# then

county_missingness = (
    master_audit[
        [
            "fips",
            "state_abbr",
            "county_name_places"
        ]
    ]
    .copy()
)

county_missingness[
    "missing_predictor_count"
] = (
    predictor_missingness_view
    .isna()
    .sum(axis=1)
)

county_missingness[
    "missing_predictor_pct"
] = (
    county_missingness[
        "missing_predictor_count"
    ]
    / predictor_missingness_view.shape[1]
    * 100
).round(2)

In [196]:
# inspect worse cases

county_missingness.sort_values(
    "missing_predictor_count",
    ascending=False
).head(25)

,fips,state_abbr,county_name_places,missing_predictor_count,missing_predictor_pct
401,09180,CT,Southeastern Connecticut,9,33.33
457,09170,CT,South Central Connecticut,9,33.33
450,09120,CT,Greater Bridgeport,9,33.33
354,09150,CT,Northeastern Connecticut,9,33.33
462,09190,CT,Western Connecticut,9,33.33
465,09160,CT,Northwest Hills,9,33.33
439,09140,CT,Naugatuck Valley,9,33.33
273,09130,CT,Lower Connecticut River Valley,9,33.33
282,09110,CT,Capitol,9,33.33
60,02195,AK,Petersburg,2,7.41


### How many counties have any predictor missingness?

In [197]:
print(
    "Counties with no missing "
    "audited predictors:",
    (
        county_missingness[
            "missing_predictor_count"
        ] == 0
    ).sum()
)

print(
    "Counties with at least one "
    "missing audited predictor:",
    (
        county_missingness[
            "missing_predictor_count"
        ] > 0
    ).sum()
)

Counties with no missing audited predictors: 3130
Counties with at least one missing audited predictor: 14


In [198]:
# much more meaningful sample-retention estimate than blindly calling dropna()

### 9.6 Missingness by state

In [199]:
# we already suspect geographic structure, summarize missingness geographically

missing_by_state = (
    county_missingness[
        county_missingness[
            "missing_predictor_count"
        ] > 0
    ]
    .groupby(
        "state_abbr"
    )
    .agg(
        counties_with_missing=(
            "fips",
            "nunique"
        ),
        total_missing_values=(
            "missing_predictor_count",
            "sum"
        )
    )
    .sort_values(
        "counties_with_missing",
        ascending=False
    )
)

missing_by_state

,counties_with_missing,total_missing_values
state_abbr,,
CT,9,81
AK,4,8
HI,1,1


This is especially useful for explaining:

> Missingness is not necessarily random; some of it is associated with geographic-definition changes.

That's relevant later when deciding whether simple median imputation is defensible.

### 9.7 Source-Level Missingness Summary

In [200]:
# create one compact table for the notebook/report

source_missingness_summary = pd.DataFrame({
    "source": [
        "PLACES target",
        "AHRF candidate features",
        "HPSA indicators/counts",
        "HPSA maximum scores",
        "SVI candidate features"
    ],

    "interpretation": [
        "Supervised outcome",
        "Potential true predictor missingness",
        "Zero indicates no qualifying designation",
        "Missing can be structural when no designation exists",
        "Check NaN and -999 sentinel values"
    ]
})

source_missingness_summary

,source,interpretation
0,PLACES target,Supervised outcome
1,AHRF candidate features,Potential true predictor missingness
2,HPSA indicators/counts,Zero indicates no qualifying designation
3,HPSA maximum scores,Missing can be structural when no designation ...
4,SVI candidate features,Check NaN and -999 sentinel values


more interpretive than numerical, but it makes the logic clear

### 9.8 Do Not Drop Complete Cases Yet
####  Missing-Data Handling Decision

No rows are removed during the audit solely because one or more predictors are missing.

A complete-case approach would risk unnecessarily reducing the county sample and could introduce geographic selection bias because some missingness is concentrated in specific geographic areas, particularly Connecticut.

Likewise, no predictor values are imputed during the audit.

Missing-data treatment will instead be determined during cleaning and feature engineering based on:

- the meaning of each variable,
- the amount of missingness,
- whether missingness is structurally expected,
- geographic concentration,
- modeling requirements, and
- sensitivity of model results to alternative handling strategies.

### Missingness Audit Result

The integrated dataset retains all 3,144 project counties and contains no missing values in the age-adjusted routine-checkup target.

Missing predictor values are not uniformly distributed across sources or geographies.

AHRF contains a small amount of true feature-level missingness, much of which is geographically structured. In particular, several healthcare-resource measures are unavailable for current Connecticut planning-region county equivalents because some underlying 2022 source data were based on the state's former county geography.

HPSA binary and count variables are complete after county aggregation. Missing maximum-score values generally represent structural missingness for counties without a qualifying HPSA designation rather than unavailable observations. These values are therefore not treated as ordinary missing data during the audit.

SVI variables were evaluated for both conventional missing values and encoded sentinel values such as `-999`.

No observations are removed and no values are imputed during this stage. Missing-data handling decisions will be made during cleaning and feature engineering, with attention to the amount, meaning, and geographic structure of missingness.

**Decision:** Retain the full 3,144-county audit dataset and defer imputation, exclusion, and structural-missingness treatment to the next processing stage.

---

## 10. Initial Harmonized Dataset

After completing source-level audits, geographic alignment, incremental merging, and missingness assessment, the validated source variables are preserved as an interim harmonized county dataset.

This dataset represents the output of the data-audit stage rather than the final modeling dataset.

At this stage:

- each row represents one U.S. county or county equivalent,
- the dataset contains all 3,144 counties in the PLACES-defined geographic universe,
- county FIPS codes are standardized,
- PLACES, AHRF, HPSA, and SVI variables have been successfully integrated,
- the supervised target is complete,
- source-level missingness has been documented, and
- no final imputation, scaling, transformation, feature engineering, or feature selection has been performed.

The harmonized dataset will serve as the input to the subsequent cleaning and feature-engineering workflow.

### 10.1 Final Audit Validation

Going to do one last structural check:

In [201]:
print("Rows:", len(master_audit))
print("Columns:", master_audit.shape[1])

print(
    "Unique FIPS:",
    master_audit["fips"].nunique()
)

print(
    "Duplicate FIPS:",
    master_audit["fips"]
    .duplicated()
    .sum()
)

print(
    "Missing target values:",
    master_audit[
        "routine_checkup_pct"
    ]
    .isna()
    .sum()
)

Rows: 3144
Columns: 41
Unique FIPS: 3144
Duplicate FIPS: 0
Missing target values: 0


In [202]:
assert len(master_audit) == 3144

assert (
    master_audit["fips"].nunique()
    == 3144
)

assert (
    master_audit["fips"]
    .duplicated()
    .sum()
    == 0
)

assert (
    master_audit[
        "routine_checkup_pct"
    ]
    .isna()
    .sum()
    == 0
)

print(
    "Final audit validation passed."
)

Final audit validation passed.


In [203]:
# checking FIPS one last time
# avoid leading zeroes

print(
    master_audit["fips"]
    .str.len()
    .value_counts()
)

fips
5    3144
Name: count, dtype: int64


In [204]:
# might as well make that an assertion

assert (
    master_audit["fips"]
    .str.len()
    .eq(5)
    .all()
)

print(
    "All county FIPS codes are "
    "five-character strings."
)

All county FIPS codes are five-character strings.


### 10.2 Review Harmonized Variables

Before exporting, inspect what is actually going into the file.

In [205]:
master_audit.columns.tolist()

['fips',
 'state_abbr',
 'county_name_places',
 'places_target_year',
 'routine_checkup_pct',
 'popn_est_22',
 'phys_nf_prim_care_pc_exc_rsdt_22',
 'np_npi_22',
 'pa_npi_22',
 'hosp_22',
 'hosp_beds_22',
 'fedly_qualfd_hlth_ctr_22',
 'rural_hlth_clincs_22',
 'rural_urban_contnm_23',
 'popn_densty_per_squr_mi_20',
 'pers_noins_lt65_pct_21',
 'pc_hpsa_count',
 'pc_hpsa_max_score',
 'pc_hpsa_any',
 'mh_hpsa_count',
 'mh_hpsa_max_score',
 'mh_hpsa_any',
 'dh_hpsa_count',
 'dh_hpsa_max_score',
 'dh_hpsa_any',
 'EP_POV150',
 'EP_UNEMP',
 'EP_HBURD',
 'EP_NOHSDP',
 'EP_UNINSUR',
 'EP_AGE65',
 'EP_AGE17',
 'EP_DISABL',
 'EP_SNGPNT',
 'EP_LIMENG',
 'EP_MINRTY',
 'EP_MUNIT',
 'EP_MOBILE',
 'EP_CROWD',
 'EP_NOVEH',
 'EP_GROUPQ']

In [206]:
# build a simple source inventory

harmonized_variable_summary = pd.DataFrame({
    "source": [
        "PLACES",
        "AHRF",
        "HPSA",
        "SVI"
    ],
    "role": [
        "Target and county identifiers",
        "Healthcare resources / access context",
        "Healthcare shortage designations",
        "Social vulnerability indicators"
    ],
    "n_variables": [
        4,  # identifiers + target, adjust if desired
        len(ahrf_feature_cols),
        len([
            col
            for col in master_audit.columns
            if "_hpsa_" in col
        ]),
        len(svi_feature_cols)
    ]
})

harmonized_variable_summary

,source,role,n_variables
0,PLACES,Target and county identifiers,4
1,AHRF,Healthcare resources / access context,11
2,HPSA,Healthcare shortage designations,9
3,SVI,Social vulnerability indicators,16


In [207]:
# mainly documentation purposes

### 10.3 Source Provenance

The harmonized dataset combines sources with different publication and measurement periods. Source provenance is recorded explicitly so that release year, measurement year, and temporal limitations remain traceable throughout the project.

In [208]:
source_manifest = pd.DataFrame({
    "source": [
        "CDC PLACES",
        "CDC/ATSDR SVI",
        "HRSA AHRF",
        "HRSA HPSA"
    ],

    "source_release": [
        "2024 release",
        "2022",
        "2023-2024 release",
        "Current extract downloaded 2026-09-25"
    ],

    "primary_project_period": [
        "2022",
        "2022",
        "Primarily 2022 candidate variables",
        "Approximate active status at 2022-12-31"
    ],

    "project_role": [
        "Routine-checkup target",
        "Social vulnerability features",
        "Healthcare resource features",
        "Healthcare shortage features"
    ],

    "important_note": [
        "Age-adjusted routine-checkup prevalence",
        "Underlying indicators used primarily for clustering",
        "Some candidate variables come from adjacent years",
        (
            "2022 designation status reconstructed from "
            "current extract; score may reflect later updates"
        )
    ]
})

source_manifest

,source,source_release,primary_project_period,project_role,important_note
0,CDC PLACES,2024 release,2022,Routine-checkup target,Age-adjusted routine-checkup prevalence
1,CDC/ATSDR SVI,2022,2022,Social vulnerability features,Underlying indicators used primarily for clust...
2,HRSA AHRF,2023-2024 release,Primarily 2022 candidate variables,Healthcare resource features,Some candidate variables come from adjacent years
3,HRSA HPSA,Current extract downloaded 2026-09-25,Approximate active status at 2022-12-31,Healthcare shortage features,2022 designation status reconstructed from cur...


### 10.4 Save Interim Dataset

The audit-stage integrated dataset is saved to `data/interim/`.

The file is intentionally labeled as an interim audit dataset rather than a final processed dataset because subsequent steps will address:

- missing-data treatment,
- rate construction,
- transformations,
- redundancy,
- feature selection,
- scaling,
- and modeling-specific preprocessing.

In [209]:
harmonized_path = (
    INTERIM_DIR
    / "master_county_audit.csv"
)

master_audit.to_csv(
    harmonized_path,
    index=False
)

print(
    "Saved harmonized dataset to:",
    harmonized_path
)

Saved harmonized dataset to: ../data/interim/master_county_audit.csv


In [210]:
harmonized_check = pd.read_csv(
    harmonized_path,
    dtype={
        "fips": "string"
    },
    low_memory=False
)

In [211]:
print(
    "Reloaded shape:",
    harmonized_check.shape
)

print(
    "Unique FIPS:",
    harmonized_check[
        "fips"
    ].nunique()
)

print(
    "FIPS lengths:"
)

print(
    harmonized_check[
        "fips"
    ]
    .str.len()
    .value_counts()
)

Reloaded shape: (3144, 41)
Unique FIPS: 3144
FIPS lengths:
fips
5    3144
Name: count, dtype: Int64


In [212]:
assert harmonized_check.shape == master_audit.shape

assert (
    harmonized_check["fips"]
    .str.len()
    .eq(5)
    .all()
)

print(
    "Saved dataset reload validation passed."
)

Saved dataset reload validation passed.


### 10.5 Save Audit Outputs

Preserving a few of the tables as possible future references.

In [213]:
# create

AUDIT_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "data_audit"
)

AUDIT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [214]:
# save main summaries: 

geographic_coverage.to_csv(
    AUDIT_RESULTS_DIR
    / "geographic_coverage.csv",
    index = False
)

merge_audit_summary.to_csv(
    AUDIT_RESULTS_DIR
    / "merge_audit_summary.csv",
    index = False
)

ahrf_missing_summary.to_csv(
    AUDIT_RESULTS_DIR
    / "ahrf_missingness.csv"
)

county_missingness.to_csv(
    AUDIT_RESULTS_DIR
    / "county_missingness.csv",
    index = False
)

source_manifest.to_csv(
    AUDIT_RESULTS_DIR
    / "source_manifest.csv",
    index = False
)

In [215]:
# SVI effective-missingness summary

svi_effective_missing.to_csv(
    AUDIT_RESULTS_DIR
    / "svi_missingness.csv"
)

In [216]:
# HPSA coverage table:

hpsa_designation_coverage.to_csv(
    AUDIT_RESULTS_DIR
    / "hpsa_designation_coverage.csv",
    index = False
)

### 10.6 Processing Deferred to the Next Stage

The harmonized audit dataset should not yet be treated as model-ready.

The following tasks are intentionally deferred:

**AHRF**
- convert selected raw counts into population-standardized rates,
- evaluate skewed distributions and possible transformations,
- resolve missing healthcare-resource values,
- assess highly correlated or conceptually overlapping measures.

**HPSA**
- determine final use of binary, count, and score variables,
- evaluate whether maximum HPSA score should be retained,
- preserve the distinction between no designation and missing/not-applicable score,
- conduct sensitivity analysis around the reconstructed 2022 designation definition.

**SVI**
- convert any documented sentinel values to missing values where appropriate,
- evaluate variable distributions and redundancy,
- determine the final clustering feature set,
- standardize variables prior to distance-based clustering.

**Model preparation**
- define supervised-learning feature groups,
- implement train/test or cross-validation preprocessing without leakage,
- scale features where required,
- perform imputation within modeling pipelines rather than before data splitting where appropriate.

These decisions will be addressed during cleaning, EDA, and modeling rather than during the source audit.

### 10.7 Initial Harmonized Dataset Summary

### Initial Harmonized Dataset Result

The source audit produced a successfully integrated county-level dataset containing 3,144 U.S. counties and county equivalents.

Each county is represented exactly once using a standardized five-character FIPS identifier. The harmonized dataset includes:

- the PLACES age-adjusted routine-checkup outcome,
- selected AHRF healthcare-resource and geographic variables,
- reconstructed county-level Primary Care, Mental Health, and Dental HPSA measures, and
- selected 2022 SVI social-vulnerability indicators.

All four data sources can therefore be integrated without loss or duplication of project counties.

The supervised-learning target is complete across the geographic universe. Remaining missingness is limited to predictor variables and includes both true source missingness and structurally expected missingness.

The dataset has been saved as:

`data/interim/master_county_audit.csv`

This file preserves source-level values after harmonization but before final feature engineering, imputation, transformation, scaling, or model-specific preprocessing.

**Decision:** `master_county_audit.csv` will serve as the controlled input to the next stage of the project.

In [217]:
### CLEAR STOP: READY FOR NEXT STEP --> CLEAN/MERGE

In [218]:
print("DATA AUDIT COMPLETE")
print("-" * 40)

print(
    f"Counties: {len(master_audit):,}"
)

print(
    f"Variables: {master_audit.shape[1]}"
)

print(
    "Target missing:",
    master_audit[
        "routine_checkup_pct"
    ].isna().sum()
)

print(
    "Duplicate FIPS:",
    master_audit[
        "fips"
    ].duplicated().sum()
)

print(
    "\nInterim dataset:"
)

print(
    harmonized_path
)

DATA AUDIT COMPLETE
----------------------------------------
Counties: 3,144
Variables: 41
Target missing: 0
Duplicate FIPS: 0

Interim dataset:
../data/interim/master_county_audit.csv
